#  1. Setup & Data Loading 📌

In [1]:
import pandas as pd
import numpy as np
import os

# Ignore warning
import warnings
warnings.filterwarnings("ignore") 

# Define the dataset path
DATA_DIR = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval'

# List all files in the directory and their sizes
print("Available files:")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        file_path = os.path.join(dirname, filename)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"{filename}: {file_size_mb:.2f} MB")

# Load the smaller datasets to inspect their structure
# We will load court_considerations.csv later since it's very large
try:
    train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
    val_df = pd.read_csv(f'{DATA_DIR}/val.csv')
    test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
    laws_df = pd.read_csv(f'{DATA_DIR}/laws_de.csv')

    print("\nData shapes:")
    print(f"train_df: {train_df.shape}")
    print(f"val_df: {val_df.shape}")
    print(f"test_df: {test_df.shape}")
    print(f"laws_df: {laws_df.shape}")

    # Display the first few rows of train data
    print("\nTrain Data Head:")
    display(train_df.head(3))

except FileNotFoundError:
    print(f"\nError: Files not found in {DATA_DIR}. Please check the exact path in your notebook.")

Available files:
sample_submission.csv: 0.00 MB
laws_de.csv: 69.59 MB
val.csv: 0.02 MB
court_considerations.csv: 2313.40 MB
train.csv: 1.88 MB
test.csv: 0.05 MB

Data shapes:
train_df: (1139, 3)
val_df: (10, 3)
test_df: (40, 2)
laws_df: (175933, 3)

Train Data Head:


,query_id,query,gold_citations
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...
2,train_0003,Das Kompetenzzentrum Völkerstrafrecht bei der ...,Art. 264m StGB


## 🔎 Insight 1

- **Massive Corpus:** The `court_considerations.csv` file being 2.3 GB is an expected challenge. Instead of loading this file directly into memory with Pandas in the future, we will likely read it in chunks or index it into a vector database/search engine (BM25, etc.).

- **Cross-lingual Reality:** Looking at the `query` column in the training set (`train_df`), the questions are in German ("Die A AG betreibt..."). However, the competition rules stated that the test data will be in English. It will be crucial for our model to establish the conceptual link between German and English.

- **Legal Articles (Laws):** `laws_df` contains 175,933 rows (legal articles/sections). This is one of the main pools from which we will be searching.

- **Target Variable Format:** In the `gold_citations` column, the citations are separated by semicolons (;). Our machine learning model will need to produce output in this exact format.

# 2. Exploratory Data Analysis - EDA 📊

In [2]:
# Calculate the number of citations per query in the training set
# We split the string by ";" and count the elements
train_df["citation_count"] = train_df["gold_citations"].apply(lambda x: len(str(x).split(";")) if pd.notnull(x) else 0)

# Display basic statistics of citation counts
print("Citation Count Statistics (Train):")
display(train_df['citation_count'].describe())

# Display the top 5 queries with the highest number of citations
print("\nTop 5 Queries with Most Citations:")
display(train_df[["query_id", "citation_count"]].sort_values(by="citation_count", ascending=False).head(5))

# Let's check the validation set to see the English queries
print("\nValidation Data Head (English Queries):")
display(val_df.head(3))

Citation Count Statistics (Train):


count    1139.000000
mean        4.090430
std         4.512206
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        44.000000
Name: citation_count, dtype: float64


Top 5 Queries with Most Citations:


,query_id,citation_count
1039,train_1040,44
43,train_0044,38
286,train_0287,38
319,train_0320,38
114,train_0115,29



Validation Data Head (English Queries):


,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...


## 🔎 Insight 2
Our outputs reveal two critical clues that will determine the outcome of the competition:

1. **The Need for Dynamic K (Dynamic Thresholding):** Looking at the statistics of our target variable, the median is 2, while the mean is ~4. This means most queries require between 1 and 5 citations. However, the maximum value is 44! This implies: We cannot simply tell our model to "always return the top 5 most likely results." If we do this, we will get too many **False Negatives** on questions requiring 44 citations. We will need to establish a threshold or a **Reranker** mechanism where our model dynamically decides the number of outputs based on its confidence score.

2. **Cross-Lingual Validation:** The `val_df` shows that the questions in our test set ("May a court lawfully order...") are asked in perfect English. However, the answers to be searched for and the legal codes are in German (Art. 221 Abs. 1 StPO). Therefore, a simple lexical matching method (like BM25 or TF-IDF) alone will not be sufficient here. We will need an advanced **Multilingual Embedding Model** (e.g., `multilingual-e5`).

# 3. Corpus Analysis & Citation Overlap 📚

In [3]:
# Get all unique citations from the training set
all_train_citations = set()
for citations in train_df['gold_citations'].dropna():
    all_train_citations.update(citations.split(';'))

print(f"Total uniqe citations in train set: {len(all_train_citations)}")

# Check how many of these are in laws_df
laws_citations = set(laws_df["citation"].unique())
overlap = all_train_citations.intersection(laws_citations)

print(f"Citations found in laws_de.csv: {len(overlap)} ")
missing_citations = len(all_train_citations) - len(overlap)
print(f"Citations likely in court_considerations.csv (or missing): {missing_citations}")

# Analyze text lengths in laws_df to understand if we need chunking
# We count the words by splitting by space
laws_df["text_word_count"] = laws_df["text"].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)
display(laws_df["text_word_count"].describe())

# Show a sample law text to see its structure
print("\nSample Law Text:")
sample_law = laws_df.iloc[0]
print(f"Citation: {sample_law["citation"]}")
print(f"Text: {sample_law["text"][:300]}...") # Show only first 300 characters

Total uniqe citations in train set: 2695
Citations found in laws_de.csv: 1876 
Citations likely in court_considerations.csv (or missing): 819


count    175933.000000
mean         31.924392
std          37.209511
min           1.000000
25%          16.000000
50%          24.000000
75%          37.000000
max        4505.000000
Name: text_word_count, dtype: float64


Sample Law Text:
Citation: Art. 1 112
Text: Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und...


## 🔎 Insight 3
We have gathered great information! Examining the output, we reach the following clear conclusions that will determine our modeling and retrieval strategy:

- **The Golden Ratio (70/30):** Approximately 70% (1,876 items) of the correct references in the training set are found within `laws_de.csv` (the laws). The remaining 30% (819 items) are most likely located in the massive `court_considerations.csv` (court decisions) file that we haven't uploaded yet. Strategically, for the first phase (baseline), it makes sense to build a search engine only on the laws. We will establish a solid foundation and obtain a score before integrating the court decisions into the system.

- **Ideal Text Lengths:** The average length of the law texts is 32 words, and the median is 24 words! This is fantastic news for us. Since the vast majority of texts are very short, we can easily fit within the 512-token limit of language models without needing complex chunking algorithms. For those rare exceptions with maximum length (4,505 words), simply applying basic truncation when feeding the model will be sufficient.

# 4. Cross-Lingual Embedding & Baseline Retrieval 🧠

In [4]:
# Install sentence-transformers if not already installed
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util
import torch
import time

# Check if GPU is available to speed up the process
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load a lightweight, multilingual embedding model
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f"Loading model '{model_name}'...")
model = SentenceTransformer(model_name, device=device)

# Take a sample of 10,000 laws to test our baseline quickly
sample_laws = laws_df.head(10000).copy()

print("\nEncoding corpus... This might take a minute.")
start_time = time.time()

# Encode the German law texts into vector embeddings 
corpus_embeddings = model.encode(sample_laws['text'].tolist(), convert_to_tensor=True, show_progress_bar=True)
print(f"Encoding finished in {time.time() - start_time:.2f} seconds.")

# Let's test with the first validation query which is in English
test_query = val_df['query'].iloc[0]
print(f"\nTest Query:\n{test_query}")

# Encode the English query
query_embedding = model.encode(test_query, convert_to_tensor=True)

# Compute cosine similarities between the query and all laws
cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

# Find the top 3 most similar laws
top_k = 3
top_results = torch.topk(cos_scores, k=top_k)

print("\n--- Top 3 Retrieval Results ---")
for score, idx in zip(top_results[0], top_results[1]):
    print(f"\nScore: {score:.4f}")
    print(f"Citation: {sample_laws.iloc[idx.item()]['citation']}")
    print(f"Text: {sample_laws.iloc[idx.item()]['text'][:200]}...") # Show a snippet

Using device: cuda
Loading model 'paraphrase-multilingual-MiniLM-L12-v2'...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Encoding corpus... This might take a minute.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Encoding finished in 5.63 seconds.

Test Query:
May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detained after an alleged late‑night assault and theft of a courier satchel containing, inter alia, €5,600—was remanded by an order dated 18 October 2024 for a maximum period up to 15 January 2025, the prosecutor sought an extension on 10 December 2024 primarily citing a concrete risk that the detainee would influence witnesses or tamper with evidence and a risk of reoffending, while the detainee opposed the extension on the ground that most witnesses have already been interviewed, the investigative steps still pending are essentially technical (phone data extraction, CCTV image analysis, and bank record checks), his release would therefore not jeopardize the inquiry, and the alleged victim has withdrawn the complaint—i.e. does the asserted concret

## 🔎 Insight 4

The incoming results highlight one of the most classic issues in Retrieval-Augmented Generation (RAG) systems: **The Semantic vs. Lexical Search Gap**.

- **What We Achieved?** The results retrieved ("arrest", "must be brought before a judge within 24 hours", "prison sentence") perfectly align with the semantic context of the query (criminal law, detention). The model understood what the query was "about."

- **What Did We Miss?** Despite the query explicitly mentioning "**Art. 221 Abs. 1 lit. b StPO**," the model returned completely different statute numbers (Art. 16, Art. 30).

- **Why Did This Happen?**
    1.  **Embedding Limitations:** Dense embedding models (like the MiniLM we downloaded) are excellent at capturing meaning but are weak in situations requiring "exact matches," such as proper nouns, numbers, or precise legal citations.
    2.  **Insufficient Data Scope:** We only indexed the first 10,000 articles. The "StPO" (Swiss Code of Criminal Procedure) we were searching for likely wasn't even within those first 10,000 rows!

# 5. Lexical Search: BM25 ⚖️

In [5]:
# Install rank_bm25 for lexical search
!pip install -q rank-bm25

from rank_bm25 import BM25Okapi
import time

print("Preparing BM25 on the FULL laws corpus...")
start_time = time.time()

# Simple tokenization: lowercasing and splitting by space 
# We fill NaN values with empty string to avoid errors
tokenized_corpus = [str(doc).lower().split() for doc in laws_df['text'].fillna("")]

# Initialize BM25 model
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 indexing finished in {time.time() - start_time:.2f} seconds.")

# Prepare the same test query 
test_query = val_df['query'].iloc[0]
tokenized_query = test_query.lower().split()

print("\nSearching with BM25...")
# Get top 3 scores and their indices 
bm25_scores = bm25.get_scores(tokenized_query)
top_k = 3
# np.argsort returns indices in ascending order, we take the last 'top_k' and reverse it
top_indices = np.argsort(bm25_scores)[-top_k:][::-1]

print("\n--- Top 3 BM25 Retrieval Results ---")
for idx in top_indices:
    score = bm25_scores[idx]
    print(f"\nScore: {score:.4f}")
    print(f"Citation: {laws_df.iloc[idx]['citation']}")
    print(f"Text: {laws_df.iloc[idx]['text'][:200]}...") # Show a snippet 

Preparing BM25 on the FULL laws corpus...
BM25 indexing finished in 3.55 seconds.

Searching with BM25...

--- Top 3 BM25 Retrieval Results ---

Score: 240.3566
Citation: Art. 3 Abs. 1 221.434
Text: 1 Die Berichterstattung über Klimabelange, die sich auf den Bericht «Recommendations of the Task Force on Climate-related Financial Disclosures» in der Fassung vom Juni 20172 und den Anhang «Implement...

Score: 189.1652
Citation: Art. 197e Abs. 2 AVO
Text: Die FINMA bezeichnet diejenigen Versicherungsgruppen, welche als international tätig gelten, und macht dies öffentlich. Sie stützt sich hierfür auf die Kriterien gemäss den Insurance Core Principles a...

Score: 161.1812
Citation: Art. 6 Abs. 2 ChKV
Text: 2 Die Nationale Behörde CWÜ ist nationale Kontaktstelle für die Organisation for the Prohibition of Chemical Weapons19 (OPCW) nach Artikel VII Absatz 4 CWÜ, für Vertragsstaaten, für Verbände sowie für...


## 🔎 Insight 5

We've encountered a classic and highly instructive "Gotcha" in data science!

- **What Happened?** BM25 retrieved completely irrelevant laws for our query, such as the "Task Force on Climate-related Financial Disclosures" or the "Organisation for the Prohibition of Chemical Weapons."

- **Why Did It Happen?** Because BM25 is a lexical (word-based) algorithm; it doesn't understand meaning. When we asked an English question, it searched Swiss/German legal texts for those English words. By chance, it found English institution names or anglicisms embedded in the German texts and presented them to us with the highest scores.

- **The Lesson:** Without performing "Query Translation," lexical matching algorithms like BM25 fail catastrophically in cross-lingual problems.

**However, there's a golden opportunity within this problem:** "Art. 221 Abs. 1 lit. b StPO." Whether in English or German, a law number is universal. Since we are building an Agentic RAG system, our agent's **first capability should be "Rule-Based Citation Extraction" from the text.**

# 6. Rule-Based Extraction with Regex 🕵️‍♂️

In [6]:
import re

# Regex pattern to catch Swiss legal citations 
# Matches patterns starting with "Art." followed by numbers, and optional "Abs." or "lit." and a law code like "StPO"
pattern = r"Art\.\s*\d+(?:\s+(?:Abs\.|lit\.|Ziff\.)\s+[a-z0-9]+)*\s+[A-Za-z]+"

print("Testing Regex on the validation query...")
test_query = val_df['query'].iloc[0]
print(f"Query snippet: {test_query[:150]}...\n")

# Extract citations using Regex
extracted_citations = re.findall(pattern, test_query)
print(f"Extracted Citations: {extracted_citations}")

# Let's check our gold citations for this query to compare
gold_cits = val_df['gold_citations'].iloc[0]
print(f"Gold Citations: {gold_cits}\n")

# Check if the extracted text exists in our laws_df citation index 
print("Checking against laws_df...")
for ext_cit in extracted_citations:
    # We will do a simple string match to see if our extracted string is part of any official citation
    matches = laws_df[laws_df['citation'].str.contains(ext_cit, regex=False, na=False)]
    print(f"Matches for '{ext_cit}': {len(matches)} found.")
    
    if len(matches) > 0:
        print(f"First match example: {matches.iloc[0]['citation']}")

Testing Regex on the validation query...
Query snippet: May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the pr...

Extracted Citations: ['Art. 221 Abs. 1 lit. b StPO']
Gold Citations: Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art. 396 Abs. 1 StPO;Art. 222 StPO;Art. 393 Abs. 1 StPO;Art. 382 Abs. 1 StPO;Art. 385 Abs. 1 StPO;Art. 221 Abs. 2 StPO;Art. 227 Abs. 1 StPO;Art. 212 Abs. 3 StPO;Art. 390 Abs. 2 StPO;Art. 422 Abs. 1 StPO;Art. 422 Abs. 2 StPO;Art. 428 Abs. 1 StPO;Art. 135 Abs. 4 StPO;Art. 100 Abs. 1 BGG;Art. 135 Abs. 3 StPO;Art. 37 Abs. 1 StBOG;Art. 39 Abs. 1 StBOG;BGE 137 IV 122 E. 6.2;BGE 137 IV 122 E. 6.4;BGE 137 IV 122 E. 4.2;BGE 132 I 21 E. 3.2;1B_210/2023 E. 4.1;BGE 132 I 21 E. 3.2.2;1B_536/2018 E. 5.1;BGE 139 IV 270 E. 3.1;BGE 133 I 168 E. 4.1;BGE 143 IV 168 E. 5.1;BGE 133 I 270 E. 3.4.2;BGE 137 IV 122 E. 4.1;BGE 132 I 21 E. 3.2.1;1B_90/2021 E. 2.1;1B_90/2021 E. 2.4;7B_496/2025 E. 

## 🔎 Insight 6

1.  **Format Mismatch (Granularity / Resolution Mismatch):** Our regex code successfully found the text "Art. 221 Abs. 1 lit. b StPO" within the question. However, when we searched for it in `laws_df`, we got zero results. Why? If you look at the "Gold Citations" list, you'll see the experts who prepared the dataset tagged this as "Art. 221 Abs. 1 StPO." This means the "lit. b" part was too detailed for the index (canonical ID) in the official database! In the future, we will need to equip our agent with a "Text Normalization" capability, teaching it to round down (trim) overly detailed citations to a higher level (paragraph level).

2.  **The Invisible Part of the Iceberg:** The question text explicitly mentioned only one law article. But take a look at the "Gold Citations" list... There are a full 42 citations! Among them are things like `BGE 137 IV 122 E. 6.2` or `1B_210/2023 E. 4.1`, which are court decisions. This means simply extracting the text within the question is not enough. We need to understand the semantic context of the question ("risk of collusion") and use it to find and retrieve the relevant precedential decisions from that massive 2.5 GB pool of court rulings.

# 7. Unveiling the Giant: Court Considerations 🏛️

In [7]:
# Load only the first 5000 rows of the massive dataset to save RAM
print("Loading a chunk of court_considerations.csv...")
court_df = pd.read_csv(f'{DATA_DIR}/court_considerations.csv', nrows=5000)

print(f"Loaded shape: {court_df.shape}")

# Display the first few rows to understand the format
print("\nCourt Considerations Head:")
display(court_df.head(3))

# Analyze text lengths to see if chunking is required for our LLM/Embedding models
court_df['text_word_count'] = court_df['text'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

print("\nCourt Text Word Count Statistics:")
display(court_df['text_word_count'].describe())

# Check for a specific pattern (e.g., 'BGE') in this small chunk
bge_count = court_df['citation'].str.contains('BGE', na=False).sum()
print(f"\nNumber of 'BGE' (Leading Decisions) in this chunk: {bge_count}")

Loading a chunk of court_considerations.csv...
Loaded shape: (5000, 2)

Court Considerations Head:


,citation,text
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents..."



Court Text Word Count Statistics:


count    5000.000000
mean      162.205200
std       134.018944
min         1.000000
25%        65.000000
50%       131.000000
75%       224.000000
max      1065.000000
Name: text_word_count, dtype: float64


Number of 'BGE' (Leading Decisions) in this chunk: 5000


## 🔎 Insight 7 

1.  **Perfect Text Size:** The average word count of court decisions is 162, with a median of 131. The maximum value is 1065 words. This is tremendous news! Modern multilingual models (e.g., multilingual-e5-base or MiniLM) can process 512 tokens at once (approximately 350-400 words). This means more than 80% of the texts will fit into our models without any truncation. We will only need to apply "Chunking" or "Truncation" for the 20% that are very long.

2.  **Sequential Data:** The first 5000 rows are all BGE (Bundesgerichtsentscheide - Swiss Federal Supreme Court Precedents). This indicates that the 2.5 GB of data is sorted by decision type or chronology, not randomly. We should keep in mind that we need to shuffle the data when splitting it (train/test split or chunking).

3.  **The Kaggle Time Limit Reality:** Considering the 12-hour Kaggle "Submission" limit, we cannot vectorize the 2.5 GB of text from scratch every time we press the "Submit" button (it would take days). We will need to pre-compute these vectors, upload them to Kaggle as a new "Dataset," and then use an ultra-fast vector database library like FAISS (Facebook AI Similarity Search) when performing the search.

# 8. Hybrid Search with RRF 🧬

In [8]:
# Let's create a sandbox of 10,000 laws to align both BM25 and Vector Search
sandbox_df = laws_df.head(10000).copy()

# 1. Prepare BM25 on the sandbox
tokenized_sandbox = [str(doc).lower().split() for doc in sandbox_df['text'].fillna("")]
bm25_sandbox = BM25Okapi(tokenized_sandbox)

# 2. Get embeddings for sandbox (We already have this from Step 4!)
# corpus_embeddings contains exactly the first 10,000 laws.

# Our English validation query
query_text = val_df['query'].iloc[0]
print("Query: (Sorgu:)")
print(query_text[:150], "...\n")

# --- A. BM25 SCORES ---
tokenized_query = query_text.lower().split()
bm25_scores = bm25_sandbox.get_scores(tokenized_query)
# Get ranks (higher score = better rank)
bm25_ranks = {idx: rank for rank, idx in enumerate(np.argsort(bm25_scores)[::-1])}

# --- B. VECTOR (DENSE) SCORES ---
query_emb = model.encode(query_text, convert_to_tensor=True)
dense_scores = util.cos_sim(query_emb, corpus_embeddings)[0].cpu().numpy()
# Get ranks 
dense_ranks = {idx: rank for rank, idx in enumerate(np.argsort(dense_scores)[::-1])}

# --- C. RECIPROCAL RANK FUSION (RRF) ---
# Formula: RRF_Score = 1 / (k + rank_bm25) + 1 / (k + rank_dense)
# (k is a constant, usually 60)
k = 60
rrf_scores = []

for idx in range(len(sandbox_df)):
    # Add 1 to rank because ranks start at 0
    rank_1 = bm25_ranks[idx] + 1
    rank_2 = dense_ranks[idx] + 1
    
    rrf_score = (1 / (k + rank_1)) + (1 / (k + rank_2))
    rrf_scores.append((idx, rrf_score))

# Sort by RRF score descending
rrf_scores.sort(key=lambda x: x[1], reverse=True)

print("--- Top 3 HYBRID Results using RRF ---")
for idx, score in rrf_scores[:3]:
    print(f"\nRRF Score: {score:.4f}")
    print(f"Citation: {sandbox_df.iloc[idx]['citation']}")
    print(f"Text: {sandbox_df.iloc[idx]['text'][:200]}...")

Query: (Sorgu:)
May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the pr ...

--- Top 3 HYBRID Results using RRF ---

RRF Score: 0.0186
Citation: Art. 29j Abs. 2 360.2
Text: 2 Wurden die Daten zu den Akten eines Strafverfahrens genommen, so werden sie auf den Auswertungsplattformen gelöscht:a. mit Eingang des rechtskräftigen Urteils;
b. mit Einstellung des Verfahrens, sow...

RRF Score: 0.0167
Citation: Art. 14a Abs. 2 362.0
Text: 2 Das Verfahren nach Absatz 1 ist bei den folgenden Ausschreibungen möglich:a. zur Festnahme zum Zweck der Auslieferung;
b. im Hinblick auf die Teilnahme an einem Strafverfahren;
c. von Vermissten ode...

RRF Score: 0.0165
Citation: Art. 3 Abs. 1 221.434
Text: 1 Die Berichterstattung über Klimabelange, die sich auf den Bericht «Recommendations of the Task Force on Climate-related Financial Disclosures» in der Fassung vom Juni 20172 und den Anhang «Implement...


## 🔎 Insight 8

Examining the results, we see the power of hybrid search (RRF): Even though we were only searching within a small pool of 10,000 rows, the model understood the query was related to criminal law ("Strafverfahren" - criminal procedure, arrest) and brought relevant articles (Art. 29j and Art. 14a). However, we still couldn't find our primary target (Art. 221 StPO).

The secret behind why we couldn't find it holds the key to the biggest trap data scientists fall into when working with legal data. Please look back at the outputs of the law data (`laws_df`) we examined in Step 3 and Step 4:

- **Citation:** Art. 1 112
- **Citation:** Art. 16 Abs. 4 362.0

Did you notice? In the official Swiss law database (our `laws_df` file), laws are not stored by their name abbreviations (ZGB, OR, StPO, StGB) but by their **SR Numbers** (Systematic Collection number)!

This means we are searching for **Art. 221 Abs. 1 StPO**, but in the database, it is actually recorded as **Art. 221 Abs. 1 312.0**! (312.0 is the code for the Swiss Code of Criminal Procedure).

Our gold citations (`gold_citations`) use abbreviations, while the corpus we are searching uses numbers. Unless we create a **Mapping Dictionary** to link these two, even the world's best AI cannot make this match.

# 9. Agent Tool: Citation Normalizer & Abbreviation Mapping 🛠️

In [9]:
# 1. Abbreviation to SR Number Mapping Dictionary 
# Note: We will add the most common ones. StPO = 312.0, StGB = 311.0, ZGB = 210, OR = 220, BGG = 173.110
law_mapping = {
    "StPO": "312.0",
    "StGB": "311.0",
    "ZGB": "210",
    "OR": "220",
    "BGG": "173.110",
    "BV": "101"
}

def normalize_citation_for_db(raw_citation):
    """
    Agent Tool: Cleans granular details and translates abbreviations to SR numbers.
    """
    # Step 1: Remove granular details like "lit. b" or "Ziff. 3"
    clean_cit = re.sub(r'\s+lit\.\s+[a-z]+', '', raw_citation)
    clean_cit = re.sub(r'\s+Ziff\.\s+\d+', '', clean_cit)
    
    # Step 2: Replace abbreviation with SR number
    for abbr, sr_num in law_mapping.items():
        if abbr in clean_cit:
            clean_cit = clean_cit.replace(abbr, sr_num)
            
    return clean_cit.strip()

# Let's test our Agent Tool on the extracted citation!
raw_extracted = 'Art. 221 Abs. 1 lit. b StPO'
normalized_citation = normalize_citation_for_db(raw_extracted)

print(f"1. Raw Extraction: '{raw_extracted}'")
print(f"2. Agent Normalized: '{normalized_citation}'")

# Step 3: Let's search this exact normalized citation in the FULL laws_df!
print(f"\nSearching the database for exactly: '{normalized_citation}'...")

exact_match = laws_df[laws_df['citation'] == normalized_citation]

if len(exact_match) > 0:
    print("\n☑️ MATCH FOUND!")
    print(f"Citation: {exact_match.iloc[0]['citation']}")
    print(f"Text Snippet: {exact_match.iloc[0]['text'][:300]}...")
else:
    print("\n✖️ No exact match. Let's try removing the 'Abs.' part as a fallback.")
    fallback_citation = re.sub(r'\s+Abs\.\s+\d+', '', normalized_citation)
    print(f"Fallback Search: '{fallback_citation}'")
    fallback_match = laws_df[laws_df['citation'] == fallback_citation]
    if len(fallback_match) > 0:
        print("☑️ FALLBACK MATCH FOUND!")
        print(f"Citation: {fallback_match.iloc[0]['citation']}")

1. Raw Extraction: 'Art. 221 Abs. 1 lit. b StPO'
2. Agent Normalized: 'Art. 221 Abs. 1 312.0'

Searching the database for exactly: 'Art. 221 Abs. 1 312.0'...

✖️ No exact match. Let's try removing the 'Abs.' part as a fallback.
Fallback Search: 'Art. 221 312.0'


## 🔎 Insight 9

- **The Mystery Solved:** The `laws_de.csv` database uses abbreviations for well-known major laws (such as Code of Criminal Procedure - StPO, Civil Code - ZGB, Code of Obligations - OR). However, for less well-known, specific sub-laws (like Art. 1 112 seen in Step 3), it uses SR numbers.

- **The Real Culprit:** So, the reason we couldn't find Art. 221 Abs. 1 lit. b StPO in Step 6 was not because of the term "StPO". The real culprit is that the experts who labeled the dataset only labeled references down to the "Paragraph" (Abs.) level, omitting the details of the "Letter" (lit.).

# 10. The Real Normalization & Exploring Law Formats ✂️

In [10]:
# The exact target from our validation gold_citations
gold_target = "Art. 221 Abs. 1 StPO"
print(f"Searching laws_df for exact gold citation: '{gold_target}'...")

# Search directly without changing StPO to an SR number
match = laws_df[laws_df['citation'] == gold_target]

if len(match) > 0:
    print("\n☑️ BOOM! Found it perfectly!")
    print(f"Citation: {match.iloc[0]['citation']}")
    print(f"Text: {match.iloc[0]['text'][:300]}...")
else:
    print("\n✖️ Not found perfectly.")

# Let's peek into the database to see how StPO citations are actually structured
print("\n--- Exploring StPO Citations in laws_df ---")
stpo_laws = laws_df[laws_df['citation'].str.contains('StPO', na=False, regex=False)]
print(f"Total 'StPO' articles found in database: {len(stpo_laws)}")

if len(stpo_laws) > 0:
    print("\nHere is a random sample of 5 StPO citations:")
    display(stpo_laws.sample(5)[['citation', 'text']])

Searching laws_df for exact gold citation: 'Art. 221 Abs. 1 StPO'...

☑️ BOOM! Found it perfectly!
Citation: Art. 221 Abs. 1 StPO
Text: 1 Untersuchungs- und Sicherheitshaft sind nur zulässig, wenn die beschuldigte Person eines Verbrechens oder Vergehens dringend verdächtig ist und ernsthaft zu befürchten ist, dass sie:a. sich durch Flucht dem Strafverfahren oder der zu erwartenden Sanktion entzieht;
b. Personen beeinflusst oder auf ...

--- Exploring StPO Citations in laws_df ---
Total 'StPO' articles found in database: 1441

Here is a random sample of 5 StPO citations:


,citation,text
131140,Art. 72 StPO,Bund und Kantone können die Zulassung sowie di...
131725,Art. 269quater Abs. 3 StPO,"3 Die Strafverfolgungsbehörde stellt sicher, d..."
131856,Art. 311 Abs. 2 StPO,2 Die Staatsanwaltschaft kann die Untersuchung...
131934,Art. 337 Abs. 3 StPO,3 Beantragt sie eine Freiheitsstrafe von mehr ...
132012,Art. 362 Abs. 2 StPO,2 Sind die Voraussetzungen für ein Urteil im a...


## 🔎 Insight 10

This little detective game proved how vital visually inspecting the data (EDA) is, whether in competitions like Kaggle or in real-world projects.

- **We Found Our Golden Rule:** In our Swiss law database (especially for major legislations), citations only go down to the "Absatz" (Abs. - paragraph) level at most. If a question or an LLM-generated answer contains deeper details like "lit. b" (subparagraph) or "Ziff. 3" (item), we need to truncate these tails to achieve an "Exact Match" in the database.

- **First Piece of the Hybrid System Complete:** We now have the rule set to convert legal articles found in questions (or draft answers the LLM might generate in the future) into database IDs (citations) with 100% accuracy.

# 11. Building the Rule-Based Extractor Pipeline ⚙️

In [11]:
def extract_and_clean_citations(text):
    """
    Agent Tool: Extracts citations from text and cleans granular details.
    """
    if not isinstance(text, str):
        return []
    
    # Base pattern to catch "Art. X ... Law"
    pattern = r"Art\.\s*\d+(?:\s+(?:Abs\.|lit\.|Ziff\.)\s+[a-z0-9]+)*\s+[A-Za-z]+"
    raw_matches = re.findall(pattern, text)
    
    cleaned_citations = set()
    for match in raw_matches:
        # Remove "lit. x" and "Ziff. x"
        clean_cit = re.sub(r'\s+lit\.\s+[a-z]+', '', match)
        clean_cit = re.sub(r'\s+Ziff\.\s+\d+', '', clean_cit)
        
        # Clean up any extra spaces
        clean_cit = " ".join(clean_cit.split())
        cleaned_citations.add(clean_cit)
        
    return list(cleaned_citations)

print("Evaluating Rule-Based Extraction on Validation Set...")

total_gold = 0
total_extracted = 0
total_hits = 0

for index, row in val_df.iterrows():
    query_text = row['query']
    
    # Handle possible NaN values in gold_citations
    gold_str = str(row['gold_citations']) if pd.notnull(row['gold_citations']) else ""
    gold_cits = set(gold_str.split(';')) if gold_str else set()
    
    # Extract using our new tool
    extracted_cits = set(extract_and_clean_citations(query_text))
    
    # Calculate hits
    hits = extracted_cits.intersection(gold_cits)
    
    total_gold += len(gold_cits)
    total_extracted += len(extracted_cits)
    total_hits += len(hits)
    
    # Print the first 2 queries as examples
    if index < 2:
        print(f"\n--- Query {index + 1} ---")
        print(f"Extracted: {extracted_cits}")
        print(f"Hits: {hits}")
        print(f"Gold Count: {len(gold_cits)}")

# Calculate basic recall
recall = (total_hits / total_gold) * 100 if total_gold > 0 else 0
print(f"\n--- Overall Validation Performance ---")
print(f"Total Gold Citations to find: {total_gold}")
print(f"Total Citations Extracted: {total_extracted}")
print(f"Total Exact Hits: {total_hits}")
print(f"Rule-Based Recall: {recall:.2f}%")

Evaluating Rule-Based Extraction on Validation Set...

--- Query 1 ---
Extracted: {'Art. 221 Abs. 1 StPO'}
Hits: {'Art. 221 Abs. 1 StPO'}
Gold Count: 42

--- Query 2 ---
Extracted: {'Art. 17 LAI'}
Hits: set()
Gold Count: 36

--- Overall Validation Performance ---
Total Gold Citations to find: 251
Total Citations Extracted: 8
Total Exact Hits: 3
Rule-Based Recall: 1.20%


## 🔎 Insight 11

This table reveals two critical insights:

1.  **The Question Does Not Contain the Answer:** People asking a legal question (Query) typically do not know the specific legal articles contained within the answer. So far, we have only extracted the citations that accidentally leaked into the question itself (a total of 8). The remaining 248 citations are hidden within the search pool, waiting for us. This means we should not be doing "Extraction," but "Retrieval."

2.  **The Multilingual Law Hell (Query 2):** In the second query, our agent found something called "Art. 17 LAI," but it did not match any of the Gold citations. Do you know why? Because Switzerland has three official languages! "LAI," a French abbreviation in the English query (Loi sur l’assurance-invalidité), is referred to as "IVG" (Invalidenversicherungsgesetz) in the German database. Our simple dictionary lookup cannot account for this.

# 12. Enter the LLM: Query Expansion & HyDE 🧠

In [12]:
# Install required libraries for LLM 
!pip install -q accelerate transformers

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# Load a highly efficient, multilingual open-source LLM
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading Agent Brain: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # Use half-precision to save memory
    device_map="auto" # Automatically put it on the P100 GPU
)

# Create a text generation pipeline
agent_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=100)

print("\nModel loaded! Testing Query Expansion...")

# Our hard English query
test_query = val_df['query'].iloc[0]
print(f"Original Query: {test_query[:150]}...")

# Agent Prompt Design
# We instruct the model to act as a Swiss legal expert 
system_prompt = "You are an expert Swiss lawyer. Your task is to extract the core legal concepts from the English query and translate them into German keywords for a database search. Also, list any relevant Swiss law abbreviations (like StPO, ZGB, OR, StGB)."
user_message = f"Query: {test_query}\n\nProvide the German keywords and Law abbreviations only:"

# Format the prompt using Qwen's chat template
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Generate the response
outputs = agent_pipe(prompt, do_sample=False)
generated_text = outputs[0]["generated_text"][len(prompt):]

print("\n🤖 Agent's Output:")
print(generated_text.strip())

Loading Agent Brain: Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model loaded! Testing Query Expansion...
Original Query: May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the pr...

🤖 Agent's Output:
**German Keywords:**
- Verfahrensverlängerung
- Proportionale Handlung
- Gefährdung durch Kollaboration
- Rechtmäßigkeit der Verlängerung
- Anhaltspunkte für Kollaboration
- Voraussetzungen zur Verlängerung

**Law Abbreviations:**
- StPO (Ständige Pflichtordnung)
- ZGB (Zivilgerichtsbarkeit)


## 🔎 Insight 12

- **Context Understanding (Success):** Our model perfectly understood the context of the issue. It successfully translated the English concept of "extension of pre-trial detention" into highly valuable German keywords such as `Verfahrensverlängerung` (extension of proceedings) and `Rechtmäßigkeit der Verlängerung` (legality of the extension).

- **Hallucination (Failure):** Because we used a relatively small 1.5 Billion parameter (1.5B) model, it hallucinated the expansion of an acronym! The actual expansion of "StPO" is `Strafprozessordnung` (Code of Criminal Procedure), but the model invented a completely fabricated word, `Ständige Pflichtordnung`.

- **Solution:** Small LLMs have limited knowledge storage capacity, but their ability to build semantic bridges between languages (reasoning) is strong. We don't need these fabricated acronym expansions; the conceptual German keywords the model generated will be more than sufficient to feed our search engine (Retriever)!

# 13. Integrating HyDE into Hybrid Search 🚀

In [14]:
print("Reloading Embedding Model...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

# 1. Combine the original English query with the LLM's German output
llm_german_keywords = generated_text.strip()
expanded_query = f"{test_query} {llm_german_keywords}"

# 2. Tokenize the expanded query for BM25
tokenized_expanded_query = expanded_query.lower().split()
print("\nRunning BM25 with expanded query...")
bm25_scores_exp = bm25_sandbox.get_scores(tokenized_expanded_query)
bm25_ranks_exp = {idx: rank for rank, idx in enumerate(np.argsort(bm25_scores_exp)[::-1])}

# 3. Encode the expanded query for Dense Retrieval using the CORRECT model
print("Running Dense Vector Search with expanded query...")
query_emb_exp = embedding_model.encode(expanded_query, convert_to_tensor=True)
dense_scores_exp = util.cos_sim(query_emb_exp, corpus_embeddings)[0].cpu().numpy()
dense_ranks_exp = {idx: rank for rank, idx in enumerate(np.argsort(dense_scores_exp)[::-1])}

# 4. Apply Reciprocal Rank Fusion (RRF)
print("Calculating RRF Scores...\n")
k = 60
rrf_scores_exp = []

for idx in range(len(sandbox_df)):
    rank_1 = bm25_ranks_exp[idx] + 1
    rank_2 = dense_ranks_exp[idx] + 1
    
    rrf_score = (1 / (k + rank_1)) + (1 / (k + rank_2))
    rrf_scores_exp.append((idx, rrf_score))

# Sort by new RRF score
rrf_scores_exp.sort(key=lambda x: x[1], reverse=True)

print("--- Top 3 HYBRID Results using Expanded Query ---")
for idx, score in rrf_scores_exp[:3]:
    print(f"\nRRF Score (RRF Skoru): {score:.4f}")
    print(f"Citation (Atıf): {sandbox_df.iloc[idx]['citation']}")
    print(f"Text (Metin): {sandbox_df.iloc[idx]['text'][:200]}...")

Reloading Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running BM25 with expanded query...
Running Dense Vector Search with expanded query...
Calculating RRF Scores...

--- Top 3 HYBRID Results using Expanded Query ---

RRF Score (RRF Skoru): 0.0183
Citation (Atıf): Art. 29j Abs. 2 360.2
Text (Metin): 2 Wurden die Daten zu den Akten eines Strafverfahrens genommen, so werden sie auf den Auswertungsplattformen gelöscht:a. mit Eingang des rechtskräftigen Urteils;
b. mit Einstellung des Verfahrens, sow...

RRF Score (RRF Skoru): 0.0173
Citation (Atıf): Art. 14a Abs. 2 362.0
Text (Metin): 2 Das Verfahren nach Absatz 1 ist bei den folgenden Ausschreibungen möglich:a. zur Festnahme zum Zweck der Auslieferung;
b. im Hinblick auf die Teilnahme an einem Strafverfahren;
c. von Vermissten ode...

RRF Score (RRF Skoru): 0.0171
Citation (Atıf): Art. 16 Abs. 4 362.0
Text (Metin): 4 Vorbehalten bleiben die speziell geregelten Verfahren für Ausschreibungen zur Festnahme zum Zweck der Auslieferung (Art. 24 und 25)....


## 🔎 Insight 13

Remember in Step 8, to speed up the process, we only took the first 10,000 rows (`sandbox_df = laws_df.head(10000)`)? The Swiss law database (`laws_df`) is likely sorted by systematic number (SR) or another order. Looking at the results, we see laws like 360.2, 362.0. The specific law we were searching for, the Code of Criminal Procedure (StPO), wasn't even in those first 10,000 rows!

**The Real-World Test:** Despite having "incomplete" data, our system (Agent + RRF) perfectly understood the context (criminal case, arrest) and retrieved the most logical alternatives available within those 10,000 rows. This means our architecture works flawlessly; our search space was simply too narrow!

# 14. Scaling Up: FAISS & The Full Arsenal 🚀

In [17]:
# Install FAISS for ultra-fast vector search
!pip install -q faiss-cpu

import faiss

print("Encoding the FULL laws corpus (175k rows)... This might take 2-3 minutes on P100. Grab a coffee! ☕")

start_time = time.time()
# Encode the entire laws_df
full_corpus_embeddings = embedding_model.encode(laws_df['text'].fillna("").tolist(), show_progress_bar=True)
full_corpus_embeddings = np.array(full_corpus_embeddings).astype('float32')
print(f"Full encoding finished in {time.time() - start_time:.2f} seconds.")

# 1. Build FAISS CPU Index
embedding_dim = full_corpus_embeddings.shape[1]
# Normalize vectors to use Inner Product as Cosine Similarity
faiss.normalize_L2(full_corpus_embeddings) 
index = faiss.IndexFlatIP(embedding_dim)
index.add(full_corpus_embeddings)
print(f"\nFAISS CPU Index built with {index.ntotal} vectors!")

# 2. Search with our Expanded Query on the FULL Dataset!
print(f"\nSearching for: {expanded_query[:100]}...")

# Vector Search via FAISS CPU
query_emb_full = embedding_model.encode([expanded_query])
query_emb_full = np.array(query_emb_full).astype('float32')
faiss.normalize_L2(query_emb_full)

top_k_search = 100 # Retrieve top 100 for RRF fusion
dense_scores_full, dense_indices_full = index.search(query_emb_full, top_k_search)

# BM25 Search on FULL dataset
tokenized_expanded_query = expanded_query.lower().split()
bm25_scores_full = bm25.get_scores(tokenized_expanded_query)
bm25_indices_full = np.argsort(bm25_scores_full)[-top_k_search:][::-1]

# 3. Apply RRF on the FULL results
rrf_scores_full = {}
k_rrf = 60

# Add Dense ranks
for rank, idx in enumerate(dense_indices_full[0]):
    rrf_scores_full[idx] = rrf_scores_full.get(idx, 0) + (1 / (k_rrf + rank + 1))

# Add BM25 ranks
for rank, idx in enumerate(bm25_indices_full):
    rrf_scores_full[idx] = rrf_scores_full.get(idx, 0) + (1 / (k_rrf + rank + 1))

# Sort final RRF scores
sorted_rrf_full = sorted(rrf_scores_full.items(), key=lambda item: item[1], reverse=True)

print("\n--- Top 5 HYBRID Results on FULL Database ---")
for idx, score in sorted_rrf_full[:5]:
    print(f"\nRRF Score: {score:.4f}")
    print(f"Citation: {laws_df.iloc[idx]['citation']}")
    print(f"Text Snippet: {laws_df.iloc[idx]['text'][:200]}...")

Encoding the FULL laws corpus (175k rows)... This might take 2-3 minutes on P100. Grab a coffee! ☕


Batches:   0%|          | 0/5498 [00:00<?, ?it/s]

Full encoding finished in 132.35 seconds.

FAISS CPU Index built with 175933 vectors!

Searching for: May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit....

--- Top 5 HYBRID Results on FULL Database ---

RRF Score: 0.0164
Citation: Art. 352 Abs. 1 StPO
Text Snippet: 1 Hat die beschuldigte Person im Vorverfahren den Sachverhalt eingestanden oder ist dieser anderweitig ausreichend geklärt, so erlässt die Staatsanwaltschaft einen Strafbefehl, wenn sie, unter Einrech...

RRF Score: 0.0164
Citation: Art. 3 Abs. 1 221.434
Text Snippet: 1 Die Berichterstattung über Klimabelange, die sich auf den Bericht «Recommendations of the Task Force on Climate-related Financial Disclosures» in der Fassung vom Juni 20172 und den Anhang «Implement...

RRF Score: 0.0161
Citation: Art. 2 Abs. 5 BStKR
Text Snippet: 5 Erlässt die Bundesanwaltschaft eine Nichtanhandnahmeverfügung (Art. 310 der Strafprozessordnung6, StPO), stellt sie das Verfahren ein (Art. 319 f

## 🔎 Insight 14

- **Incredible Progress:** If you recall, in Step 5, our system was retrieving completely irrelevant laws like "Chemical Weapons," etc. Now, when we look at the results, we see articles directly from the Code of Criminal Procedure (StPO), such as Art. 352 Abs. 1 StPO and Art. 82 Abs. 1 StPO! Our model has perfectly grasped the context of the problem (criminal law, arrest, trial) and is navigating within the pages of the correct legal code.

- **BM25 Noise:** Results like 221.434 (Climate reporting) and AVO are still in the list. Why? Because the long, specific words in the original English question (e.g., "Task Force," "Financial") are still tricking BM25.

- **Major Miss:** Most importantly, our primary target, Art. 221 Abs. 1 StPO, is not in the top 5! The vector mathematics and BM25 could not focus enough on the specific number "Art. 221."

# 15. The Master Agent: Unifying the Pipeline 🎼

In [18]:
def agentic_retriever(query_text, top_k_hybrid=5):
    """
    The Ultimate Agentic Retrieval Pipeline!
    """
    final_citations = set()
    
    # ---------------------------------------------------------
    # TOOL 1: Rule-Based Exact Match (Regex)
    # ---------------------------------------------------------
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    
    # Add exact matches directly to final results with absolute confidence!
    final_citations.update(exact_matches)
    
    # ---------------------------------------------------------
    # TOOL 2: LLM Query Expansion + Hybrid Search 
    # ---------------------------------------------------------
    # Note: To save time in this test, we are using the already generated German keywords.
    # In a full loop, we would call the LLM here for every query.
    expanded_query_local = f"{query_text} {llm_german_keywords}"
    
    # 2a. Dense Vector Search (FAISS)
    q_emb = embedding_model.encode([expanded_query_local])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    _, dense_indices = index.search(q_emb, 100)
    
    # 2b. Sparse Search (BM25)
    tok_q = expanded_query_local.lower().split()
    bm25_scores = bm25.get_scores(tok_q)
    bm25_indices = np.argsort(bm25_scores)[-100:][::-1]
    
    # 2c. RRF Fusion
    rrf_scores = {}
    for rank, idx in enumerate(dense_indices[0]):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1 / (60 + rank + 1))
    for rank, idx in enumerate(bm25_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1 / (60 + rank + 1))
        
    sorted_rrf = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    
    # Add the top K hybrid results to our final citations
    for idx, _ in sorted_rrf[:top_k_hybrid]:
        final_citations.add(laws_df.iloc[idx]['citation'])
        
    return list(final_citations), exact_matches

# Let's test our Master Agent on our tough validation query!
tough_query = val_df['query'].iloc[0]
print("Running Master Agent...\n")

final_results, rule_based_finds = agentic_retriever(tough_query)

print("--- 🎯 MASTER AGENT RESULTS ---")
print(f"1. Caught by Rule-Based Tool: {rule_based_finds}")
print(f"2. Total Citations Retrieved: {final_results}")

# Did we hit the gold target?
gold_cits = set(val_df['gold_citations'].iloc[0].split(';'))
hits = set(final_results).intersection(gold_cits)

print(f"\n☑️ Final Hits against Gold Citations: {hits}")

Running Master Agent...

--- 🎯 MASTER AGENT RESULTS ---
1. Caught by Rule-Based Tool: ['Art. 221 Abs. 1 StPO']
2. Total Citations Retrieved: ['Art. 352 Abs. 1 StPO', 'Art. 3 Abs. 1 221.434', 'Art. 221 Abs. 1 StPO', 'Art. 82 Abs. 1 StPO', 'Art. 2 Abs. 5 BStKR', 'Art. 197e Abs. 2 AVO']

☑️ Final Hits against Gold Citations: {'Art. 221 Abs. 1 StPO'}


## 🔎 Insight 15

- **The Power of Tool Use:** If we had only relied on Vector Search (FAISS) and Lexical Search (BM25), we would never have found Art. 221 (as we saw in Step 14). However, we instructed our agent: "If you explicitly see a law in the text, stop guessing and use the Regex tool directly!" And the result: 100% precision.

- **Complementary Hybrid Search:** While our Regex tool definitively captured Art. 221, our Hybrid RAG engine was not idle and retrieved other relevant articles from the Code of Criminal Procedure (StPO) that fit the context. Even though they weren't among the gold citations, they demonstrate how deeply our model understands the subject matter.

# 16. The Submission Pipeline 🏆

In [20]:
from tqdm import tqdm

embedding_dim = full_corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(full_corpus_embeddings)
print("FAISS Index ready! Starting Submission Pipeline...\n")

submission_data = []

# Notice the loop variable is now 'row_idx' instead of 'index'!
for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing Test Queries"):
    query_id = row['query_id']
    query_text = row['query']
    
    # --- 1. LLM Query Expansion (HyDE) ---
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts from the English query and translate them into German keywords. List relevant Swiss law abbreviations (e.g., StPO, ZGB)."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords and Abbreviations:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    
    # Generate keywords
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    # --- 2. Rule-Based Extraction ---
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    
    final_citations = set(exact_matches) # Start with absolute confident hits
    
    # --- 3. Hybrid Search (FAISS + BM25) ---
    expanded_query = f"{query_text} {german_keywords}"
    
    # Dense (FAISS) using the new 'faiss_index' variable! 
    q_emb = embedding_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    _, dense_indices = faiss_index.search(q_emb, 50) # Top 50 for fusion
    
    # Sparse (BM25)
    tok_q = expanded_query.lower().split()
    bm25_scores = bm25.get_scores(tok_q)
    bm25_indices = np.argsort(bm25_scores)[-50:][::-1]
    
    # RRF Fusion
    rrf_scores = {}
    for rank, idx in enumerate(dense_indices[0]):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1 / (60 + rank + 1))
    for rank, idx in enumerate(bm25_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1 / (60 + rank + 1))
        
    sorted_rrf = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    
    # Add top 5 hybrid results
    for idx, _ in sorted_rrf[:5]:
        final_citations.add(laws_df.iloc[idx]['citation'])
        
    # --- 4. Format for Kaggle ---
    predicted_string = ";".join(list(final_citations))
    
    submission_data.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

# Create DataFrame and save to CSV
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('submission.csv', index=False)

print("\n☑️ submission.csv successfully created!")
display(submission_df.head(5))

FAISS Index ready! Starting Submission Pipeline...



Processing Test Queries:  20%|██        | 8/40 [02:01<08:10, 15.34s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Processing Test Queries: 100%|██████████| 40/40 [09:43<00:00, 14.58s/it]


✅ submission.csv successfully created!


,query_id,predicted_citations
0,test_001,Art. 45 Abs. 2 SPBV-EJPD;Art. 10 Abs. 3 VTE;Ar...
1,test_002,Art. 94 Abs. 4 SVG;Art. 3 Abs. 1 221.434;Art. ...
2,test_003,Art. 3 Abs. 1 STUG;Art. 11 Abs. 1 STUV;Art. 3 ...
3,test_004,Art. 3 Abs. 1 221.434;Art. 12 Abs. 1bis StV;Ar...
4,test_005,Art. 28 Abs. 2 AVV;Art. 3 Abs. 1 221.434;Art. ...


## 🔎 Insight 16
**Public Score: 0.01191**

At first glance, the Public Score (Coverage Rate / F1 Score) we received may seem very low, but this is a tremendous reference point for us. Why?

- **Main Reason for the Low Score:** So far, we have only searched within `laws_de.csv` (the law articles). However, if you recall from our EDA analysis in Step 3, we observed that a significant portion of the gold citations and nearly all of the long references come from the massive `court_considerations.csv` file (court decisions). The sole reason we are currently this low on the leaderboard is that 90% of our pool (the court decisions) is missing!

- **The Pipeline is Working:** Our system did not crash, did not give a format error, and Kaggle successfully accepted our results. This means the entire engineering infrastructure of our "Agentic RAG" architecture is solid!

# 17. Conquering the Giant: Processing Court Considerations 🏛️

In [21]:
# Re-initialize the embedding model on GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device for encoding: {device}")
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

# We will create a NEW, unified FAISS index for BOTH laws and court cases
embedding_dim = 384 # MiniLM dimension
unified_faiss_index = faiss.IndexFlatIP(embedding_dim)

# We need a unified list to keep track of citations corresponding to the FAISS index
unified_citations = []

# --- 1. Add Laws ---
print("\nAdding Laws to Unified Index...")
start_time = time.time()
laws_texts = laws_df['text'].fillna("").tolist()
laws_embeddings = embedding_model.encode(laws_texts, show_progress_bar=True)
laws_embeddings = np.array(laws_embeddings).astype('float32')
faiss.normalize_L2(laws_embeddings)
unified_faiss_index.add(laws_embeddings)

# Add law citations to our unified list
unified_citations.extend(laws_df['citation'].tolist())
print(f"Laws added! Time: {time.time() - start_time:.2f}s. Total vectors: {unified_faiss_index.ntotal}")

# --- 2. Add Court Considerations in CHUNKS ---
print("\nAdding Court Considerations (The Giant!)... This will take 50-55 minutes. Grab a coffee! ☕")

chunk_size = 50000 # Read 50k rows at a time to save RAM 
court_file_path = f'{DATA_DIR}/court_considerations.csv'
chunk_count = 0
total_court_time = 0

# Open the giant CSV in chunks
for chunk in pd.read_csv(court_file_path, chunksize=chunk_size):
    chunk_start = time.time()
    chunk_count += 1
    
    # Extract text and fill NaNs
    texts = chunk['text'].fillna("").tolist()
    citations = chunk['citation'].tolist()
    
    # Encode the chunk
    embeddings = embedding_model.encode(texts, show_progress_bar=False) # Turn off progress bar for cleaner logs
    embeddings = np.array(embeddings).astype('float32')
    faiss.normalize_L2(embeddings)
    
    # Add to FAISS index
    unified_faiss_index.add(embeddings)
    
    # Add to unified citations list
    unified_citations.extend(citations)
    
    chunk_time = time.time() - chunk_start
    total_court_time += chunk_time
    
    # Print progress every 5 chunks
    if chunk_count % 5 == 0:
        print(f"Processed chunk {chunk_count} ({chunk_count * chunk_size} rows)... Total vectors: {unified_faiss_index.ntotal}")

print(f"\n☑️ FULL DATASET INDEXED!")
print(f"Total time for Court Cases: {total_court_time / 60:.2f} minutes.")
print(f"Final FAISS Index Size: {unified_faiss_index.ntotal} vectors.")
print(f"Total Unified Citations Length: {len(unified_citations)}")

Using device for encoding: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Adding Laws to Unified Index...


Batches:   0%|          | 0/5498 [00:00<?, ?it/s]

Laws added! Time: 132.17s. Total vectors: 175933

Adding Court Considerations (The Giant!)... This will take 15-20 minutes. Grab a coffee! ☕
Processed chunk 5 (250000 rows)... Total vectors: 425933
Processed chunk 10 (500000 rows)... Total vectors: 675933
Processed chunk 15 (750000 rows)... Total vectors: 925933
Processed chunk 20 (1000000 rows)... Total vectors: 1175933
Processed chunk 25 (1250000 rows)... Total vectors: 1425933
Processed chunk 30 (1500000 rows)... Total vectors: 1675933
Processed chunk 35 (1750000 rows)... Total vectors: 1925933
Processed chunk 40 (2000000 rows)... Total vectors: 2175933
Processed chunk 45 (2250000 rows)... Total vectors: 2425933
Processed chunk 50 (2500000 rows)... Total vectors: 2652248

✅ FULL DATASET INDEXED!
Total time for Court Cases: 55.40 minutes.
Final FAISS Index Size: 2652248 vectors.
Total Unified Citations Length: 2652248


## 🔎 Insight 17

- **RAM Limit and the BM25 Danger:** Remember in Step 5, we used BM25 for keyword-based search. However, to tokenize and index the 2.6 million rows of text, the BM25 algorithm can consume all of the 30 GB of RAM provided by Kaggle, potentially crashing the Kernel (OOM - Out of Memory) and wasting an hour of our work.

- **The Power of Pure Vectors (Dense Retrieval):** We now have 2.6 million vectors in our FAISS index, and this system excels at semantic matching. To avoid risks while creating our Kaggle submission for the test data, we will proceed with only the trio of Regex (Exact Match) + LLM (Query Expansion) + FAISS (Vector Search).

# 18. The Ultimate Submission 🏆

In [22]:
print("Initiating the ULTIMATE Submission Pipeline on 2.6M documents! 🚀")

final_submission_data = []

# Loop through all 40 queries in the test set
for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing Test Queries"):
    query_id = row['query_id']
    query_text = row['query']
    
    # --- 1. Rule-Based Exact Match (Regex) ---
    # We still use laws_df for exact law extraction, as court cases don't usually appear in the query directly
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    
    final_citations = set(exact_matches) # Start with absolute confident hits
    
    # --- 2. LLM Query Expansion (HyDE) ---
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts from the English query and translate them into German keywords. List relevant Swiss law abbreviations (e.g., StPO, ZGB)."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords and Abbreviations:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    
    # Generate keywords
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    # --- 3. Dense Vector Search on the 2.6M UNIFIED Index ---
    expanded_query = f"{query_text} {german_keywords}"
    
    q_emb = embedding_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb) # Normalize for Inner Product
    
    # Let's retrieve the top 15 most relevant citations from the giant pool
    top_k_retrieve = 15 
    distances, indices = unified_faiss_index.search(q_emb, top_k_retrieve)
    
    # Add the retrieved citations to our final set using our unified list
    for idx in indices[0]:
        final_citations.add(unified_citations[idx])
        
    # --- 4. Format for Kaggle ---
    predicted_string = ";".join(list(final_citations))
    
    final_submission_data.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

# Create DataFrame and overwrite the submission.csv
final_sub_df = pd.DataFrame(final_submission_data)
final_sub_df.to_csv('submission.csv', index=False)

print("\n☑️ ULTIMATE submission.csv successfully created! Ready to conquer the leaderboard! 🏆")
display(final_sub_df.head(5))

Initiating the ULTIMATE Submission Pipeline on 2.6M documents! 🚀


Processing Test Queries: 100%|██████████| 40/40 [01:32<00:00,  2.32s/it]


✅ ULTIMATE submission.csv successfully created! Ready to conquer the leaderboard! 🏆


,query_id,predicted_citations
0,test_001,6B_673/2016 E. D;6B_442/2011 27.07.2012 E. A;4...
1,test_002,8C_144/2020 E. 5.1;6B_228/2011 06.05.2011 E. 1...
2,test_003,6B_79/2011 05.08.2011 E. 5.3;8C_241/2009 01.10...
3,test_004,2C_354/2015 E. C;4F_28/2025 E. 2;9C_542/2009 1...
4,test_005,H 297/99 29.05.2000 E. A;6B_79/2011 05.08.2011...


## 🔎 Insight 18
**Public Score:** 0.00765

We enthusiastically processed 2.5 GB of data and expanded our pool 15 times, but our score dropped from 0.01191 to 0.00765. So why did this happen? Where did we go wrong? Actually, we didn't make a mistake; we just faced a very classic Machine Learning problem:

1.  **The Needle and Haystack Problem (Noise & Distraction):** Previously, our model was searching for the answer among 175,000 rows (only laws). Now, it has a massive haystack of 2.6 million rows in front of it. Because we didn't implement advanced filtering, our model got lost among thousands of court decisions that are semantically very similar but incorrect.

2.  **Precision Massacre:** The competition's evaluation metric is the Macro F1 score. The F1 score is a balance between what you correctly find (Recall) and the false alarms you make (Precision). In our code, we set `top_k_retrieve = 15`. This commanded the model: "Bring me exactly 15 answers for every question." If a question only requires 2 citations and the model returns 15, the extra 13 become "False Positives," devastating our F1 score!

3.  **The Absence of BM25:** Due to RAM concerns, we deactivated BM25 and focused solely on semantics (vectors). Because court decision texts are long, they occupy a lot of space in the vector space and push the short, clear law articles down the list of results.

# 19. Local Validation & Score Distribution 🧪

In [24]:
print("Running Local Evaluation on Validation Set...\n")

# Let's take the first query from our validation set
test_query = val_df['query'].iloc[0]
gold_cits = set(val_df['gold_citations'].iloc[0].split(';'))

print(f"Query: {test_query[:150]}...")
print(f"Total Gold Citations to find: {len(gold_cits)}\n")

# --- 1. LLM Query Expansion ---
prompt_messages = [
    {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts from the English query and translate them into German keywords. List relevant Swiss law abbreviations (e.g., StPO, ZGB)."},
    {"role": "user", "content": f"Query: {test_query}\n\nKeywords and Abbreviations:"}
]
prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
german_keywords = outputs[0]["generated_text"][len(prompt):].strip()

# --- 2. Dense Search on 2.6M Index ---
expanded_query = f"{test_query} {german_keywords}"
q_emb = embedding_model.encode([expanded_query])
q_emb = np.array(q_emb).astype('float32')
faiss.normalize_L2(q_emb)

# Get top 15 to analyze the score drop-off
top_k = 15
scores, indices = unified_faiss_index.search(q_emb, top_k)

print("--- FAISS Retrieval Scores ---")
print("Score | Is Gold? | Citation")
print("-" * 50)

hits_found = 0

# Loop through the results and check if they are in the gold standard
for i in range(top_k):
    score = scores[0][i]
    idx = indices[0][i]
    citation = unified_citations[idx]
    
    is_gold = "☑️ YES" if citation in gold_cits else "✖️ NO"
    if is_gold == "☑️ YES":
        hits_found += 1
        
    print(f"{score:.4f} | {is_gold:6} | {citation}")

print("-" * 50)
print(f"Total Hits in Top {top_k}: {hits_found}")

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Local Evaluation on Validation Set...

Query: May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the pr...
Total Gold Citations to find: 42

--- FAISS Retrieval Scores ---
Score | Is Gold? | Citation
--------------------------------------------------
0.8379 | ✖️ NO  | 7B_910/2025 E. B
0.8332 | ✖️ NO  | 7B_58/2025 07.02.2025 E. 3.3
0.8309 | ✖️ NO  | 7B_984/2023 E. 3.3.2
0.8284 | ✖️ NO  | 7B_203/2024 E. 3.1
0.8276 | ✖️ NO  | 1B_202/2017 E. 2.1
0.8251 | ✖️ NO  | 7B_645/2023 E. 3.2.2
0.8246 | ✖️ NO  | 7B_203/2024 E. 3.2
0.8235 | ✖️ NO  | 1B_191/2011 17.05.2011 E. 2
0.8232 | ✖️ NO  | BGE 137 IV 177 E. 2.1
0.8225 | ✖️ NO  | 1P.147/2000 31.03.2000 E. 1
0.8202 | ✖️ NO  | 1B_589/2021 E. 2
0.8201 | ✖️ NO  | 1B_80/2010 06.04.2010 E. 5
0.8193 | ✖️ NO  | 6B_169/2012 25.06.2012 E. 5
0.8170 | ✖️ NO  | 1B_384/2018 E. 3.1
0.8169 | ✖️ NO  | 7B_1243/2024 E. 2.2
--------------------------------------------

## 🔎 Insight 19

Behind the "0" hits and the score drop in this table lie two major "Monsters" frequently encountered in RAG (Retrieval-Augmented Generation) systems:

- **Length Bias:** Court decisions (*court_considerations*) are like epics, consisting of hundreds of words. Laws (*laws*) are short, clear rules of 20-30 words. Vector models (like MiniLM) tend to always find longer texts with more context more "similar" than short texts. The result? The moment we entered the 2.6 million pool, those lengthy court decisions crushed our short legal articles, burying them to the bottom of the list (into invisibility)! Our golden law, Art. 221 Abs. 1 StPO, drowned in the vector space.

- **Semantic Clones:** Look at the court decisions it retrieved: 7B_910/2025, BGE 137 IV 177, etc... ALL of these decisions are about the "extension of pre-trial detention." Semantically, they are all correct, but they are not the specific precedent decisions (Gold Citations) that Kaggle asked for. Our model isn't looking for a needle in a haystack; it's finding other pieces of hay that look like hay in the haystack.

**Solution: Federated Search**
We will not throw all the data into the same pool and confuse the AI. We will split the system:

1.  **Agent 1 (The Detective):** Will only catch precisely mentioned laws using rule-based Regex.
2.  **Agent 2 (The Law Expert):** Will perform BM25 + Vector search only in the laws pool (*laws_df*). (This way, laws won't be crushed under the weight of court decisions).
3.  **Agent 3 (The Precedent Expert):** Will perform vector search only in the large pool of court decisions.

Finally, we will hybridize the results brought in by these three agents!

# 20. The Federated Search Pipeline 🔀

In [25]:
print("Initiating the FEDERATED Search Pipeline! 🚀")

federated_submission_data = []

# Ensure we have our BM25 index and FAISS index for laws ready
# faiss_index = Laws only (175k)
# unified_faiss_index = Everything (2.6M)

for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing Federated Queries"):
    query_id = row['query_id']
    query_text = row['query']
    
    final_citations = set()
    
    # --- ROUTE 1: Rule-Based Exact Law Match 
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    final_citations.update(exact_matches)
    
    # --- ROUTE 2: LLM Query Expansion ---
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts from the English query and translate them into German keywords. List relevant Swiss law abbreviations (e.g., StPO, ZGB)."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords and Abbreviations:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    expanded_query = f"{query_text} {german_keywords}"
    
    # Vectorize the expanded query
    q_emb = embedding_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    # --- ROUTE 3: Law Search  ---
    # Retrieve top 5 laws strictly to prevent them from being buried by court cases
    _, dense_indices_laws = faiss_index.search(q_emb, 10)
    
    tok_q = expanded_query.lower().split()
    bm25_scores = bm25.get_scores(tok_q)
    bm25_indices = np.argsort(bm25_scores)[-10:][::-1]
    
    rrf_scores_laws = {}
    for rank, idx in enumerate(dense_indices_laws[0]):
        rrf_scores_laws[idx] = rrf_scores_laws.get(idx, 0) + (1 / (60 + rank + 1))
    for rank, idx in enumerate(bm25_indices):
        rrf_scores_laws[idx] = rrf_scores_laws.get(idx, 0) + (1 / (60 + rank + 1))
        
    sorted_rrf_laws = sorted(rrf_scores_laws.items(), key=lambda item: item[1], reverse=True)
    
    # Add top 3 strictly from laws
    for idx, _ in sorted_rrf_laws[:3]:
        final_citations.add(laws_df.iloc[idx]['citation'])
        
    # --- ROUTE 4: Court Case Search ---
    # Retrieve top 5 decisions from the unified index
    _, dense_indices_unified = unified_faiss_index.search(q_emb, 10)
    
    # Add to final citations, but only if they contain 'BGE' or '_' (Typical court case formats)
    added_courts = 0
    for idx in dense_indices_unified[0]:
        citation = unified_citations[idx]
        if 'BGE' in citation or '_' in citation or '/' in citation:
            final_citations.add(citation)
            added_courts += 1
        if added_courts >= 5: # Limit court cases to top 5 to keep precision high
            break

    # --- FINAL: Format for Kaggle ---
    predicted_string = ";".join(list(final_citations))
    
    federated_submission_data.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

# Save the new strategic submission 
federated_sub_df = pd.DataFrame(federated_submission_data)
federated_sub_df.to_csv('submission_federated.csv', index=False)

print("\n☑️ submission_federated.csv successfully created!")
display(federated_sub_df.head(5))

Initiating the FEDERATED Search Pipeline! 🚀


Processing Federated Queries: 100%|██████████| 40/40 [10:11<00:00, 15.29s/it]


☑️ submission_federated.csv successfully created!


,query_id,predicted_citations
0,test_001,4A_575/2018 E. A;2C_154/2012 05.09.2012 E. A;6...
1,test_002,8C_144/2020 E. 5.1;U 282/02 10.02.2004 E. A;Ar...
2,test_003,Art. 3 Abs. 1 STUG;Art. 11 Abs. 1 STUV;Art. 3 ...
3,test_004,2C_354/2015 E. C;4F_28/2025 E. 2;Art. 3 Abs. 1...
4,test_005,6B_79/2011 05.08.2011 E. 5.3;6B_841/2016 E. 1....


## 🔎 Insight 20

Why couldn't our score fully surpass the initial baseline? Because the evaluation metric used by Kaggle, the Macro F1 Score, never forgives "False Positives." The mathematics behind the F1 score is as follows:

In our final code, we instructed the model: "Regardless of whether you are certain or not, force 5 court decisions for every query." If the query is a simple question about a law that doesn't require any court decisions at all, those 5 court decisions brought by the model are considered "False Positives," completely devastating our "Precision" score.

# 21. The Reranker 🎯

In [26]:
from sentence_transformers import CrossEncoder

print("Loading Multilingual Cross-Encoder (Reranker)...")
# This is a specialized model trained to detect exact relevance between query and document
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', max_length=512, device=device)

print("Reranker loaded! Initiating the TWO-STAGE Retrieval Pipeline! 🚀")

reranked_submission_data = []

for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Reranking Queries"):
    query_id = row['query_id']
    query_text = row['query']
    
    final_citations = set()
    
    # --- STAGE 1: Fast Retrieval ---
    
    # 1. Regex Exact Matches
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    final_citations.update(exact_matches)
    
    # 2. LLM Expansion
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts and translate to German keywords. List abbreviations."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    expanded_query = f"{query_text} {german_keywords}"
    q_emb = embedding_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    # Fetch Top 15 Candidates from the Giant Unified Index
    top_k_retrieve = 15
    _, unified_indices = unified_faiss_index.search(q_emb, top_k_retrieve)
    
    # Prepare candidate texts for the Reranker
    candidate_citations = []
    cross_encoder_pairs = []
    
    for idx in unified_indices[0]:
        cit = unified_citations[idx]
        
        # Find the text of this citation
        # It could be in laws_df or court_df. Since we don't have court_df fully in RAM, 
        # we stored it in FAISS. For the sake of RAM, let's fetch it smartly.
        if cit in laws_df['citation'].values:
            text = laws_df[laws_df['citation'] == cit].iloc[0]['text']
        else:
            # If it's a court case, we need to read it. But to avoid RAM crashes, 
            # we will just pass the citation ID itself + german keywords as a proxy for relevance
            text = cit 
            
        candidate_citations.append(cit)
        cross_encoder_pairs.append([expanded_query, text])
    
    # --- STAGE 2: Reranking ---
    if cross_encoder_pairs:
        # Get relevance scores
        rerank_scores = reranker.predict(cross_encoder_pairs)
        
        # Only keep candidates with a POSITIVE score (Dynamic Thresholding!)
        for cit, score in zip(candidate_citations, rerank_scores):
            # The mmarco model outputs logits. Scores > 0 generally mean "relevant"
            if score > 0.0: 
                final_citations.add(cit)
                
    # --- FINAL: Format for Kaggle ---
    predicted_string = ";".join(list(final_citations))
    
    reranked_submission_data.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

# Save the ultimate reranked submission
reranked_sub_df = pd.DataFrame(reranked_submission_data)
reranked_sub_df.to_csv('submission_reranked.csv', index=False)

print("\n☑️ submission_reranked.csv successfully created!")
display(reranked_sub_df.head(5))

Loading Multilingual Cross-Encoder (Reranker)...


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

README.md: 0.00B [00:00, ?B/s]

Reranker loaded! Initiating the TWO-STAGE Retrieval Pipeline! 🚀


Reranking Queries: 100%|██████████| 40/40 [01:36<00:00,  2.42s/it]


☑️ submission_reranked.csv successfully created!


,query_id,predicted_citations
0,test_001,4A_575/2018 E. A;4A_203/2021 E. A
1,test_002,8C_144/2020 E. 5.1;6B_228/2011 06.05.2011 E. 1...
2,test_003,6B_79/2011 05.08.2011 E. 5.3;8C_241/2009 01.10...
3,test_004,2C_354/2015 E. C;4F_28/2025 E. 2;9C_542/2009 1...
4,test_005,H 297/99 29.05.2000 E. A;6B_79/2011 05.08.2011...


## 21.1. Hacker Bypass: The Dummy Citation 🥷

In [30]:
print("Initiating Kaggle Parser Bypass Protocol...🥷")

# Load our previously reranked file
df = pd.read_csv('submission_reranked.csv')

# The dummy citation we know 100% exists and is correctly formatted
dummy_citation = "Art. 221 Abs. 1 StPO"

# Count how many empty predictions we have 
empty_mask = df['predicted_citations'].isna() | (df['predicted_citations'] == "")
empty_count = empty_mask.sum()

print(f"Found {empty_count} completely empty predictions.")
print(f"Injecting dummy citation '{dummy_citation}' to bypass the host's NaN bug...")

# Replace all NaNs and empty strings with our dummy citation
df.loc[empty_mask, 'predicted_citations'] = dummy_citation

# Save the final bypass file
# We don't even need the quote trick anymore because there are no empty strings!
df.to_csv('submission_hacker_bypass.csv', index=False)

print("\n☑️ 'submission_hacker_bypass.csv' successfully created!")

# Show the fixed rows
display(df[empty_mask].head(3))

Initiating Kaggle Parser Bypass Protocol...🥷
Found 11 completely empty predictions.
Injecting dummy citation 'Art. 221 Abs. 1 StPO' to bypass the host's NaN bug...

☑️ 'submission_hacker_bypass.csv' successfully created!


,query_id,predicted_citations
5,test_006,Art. 221 Abs. 1 StPO
8,test_009,Art. 221 Abs. 1 StPO
9,test_010,Art. 221 Abs. 1 StPO


## 🔎 Insight 21
**public score 0.01047**

There is a very clear and mathematical reason why our score falls slightly below Federated Search's (0.01086):

1.  **The Penalty of the Dummy:** As you may recall, to prevent the Kaggle bot from crashing, we intentionally placed Art. 221 Abs. 1 StPO (Detention) citations in 11 questions. The actual subject of those 11 questions could have been tax, divorce, or a traffic ticket! We knowingly created 11 "False Positives," sabotaging the Precision leg of our Macro F1 score with our own hands.

2.  **Excessively Strict Reranker:** We instructed our model: "Only pass through those with a score above 0.0." The model was so strict that it also filtered out some potentially correct citations (False Negatives).

# 22. The Golden Ratio: Top-K Reranking 🏆

In [31]:
print("Initiating the GOLDEN RATIO Reranker Pipeline! 🚀")

golden_submission_data = []

for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Optimizing Predictions"):
    query_id = row['query_id']
    query_text = row['query']
    
    final_citations = set()
    
    # 1. Exact Match via Regex
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    final_citations.update(exact_matches)
    
    # 2. LLM Query Expansion
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts and translate to German keywords. List abbreviations."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    # 3. Fast Retrieval (FAISS on 2.6M Index)
    expanded_query = f"{query_text} {german_keywords}"
    q_emb = embedding_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    top_k_retrieve = 15
    _, unified_indices = unified_faiss_index.search(q_emb, top_k_retrieve)
    
    candidate_citations = []
    cross_encoder_pairs = []
    
    for idx in unified_indices[0]:
        cit = unified_citations[idx]
        if cit in laws_df['citation'].values:
            text = laws_df[laws_df['citation'] == cit].iloc[0]['text']
        else:
            text = cit # Use citation as context for court cases
            
        candidate_citations.append(cit)
        cross_encoder_pairs.append([expanded_query, text])
    
    # 4. Reranking with Dynamic Top-K
    if cross_encoder_pairs:
        rerank_scores = reranker.predict(cross_encoder_pairs)
        
        # Zip candidates with their scores and sort them descending
        scored_candidates = list(zip(candidate_citations, rerank_scores))
        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        
        # THE FIX: Take exactly the top 3 best results, regardless of absolute score.
        # This guarantees NO EMPTY strings and high precision!
        for cit, score in scored_candidates[:3]:
            final_citations.add(cit)
                
    # Format for Kaggle
    predicted_string = ";".join(list(final_citations))
    
    golden_submission_data.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

# Save the final optimized submission
golden_sub_df = pd.DataFrame(golden_submission_data)

# Just a safety net: in the 0.0001% chance something is empty, fallback to a generic law
golden_sub_df['predicted_citations'] = golden_sub_df['predicted_citations'].replace("", "Art. 1 ZGB") 

golden_sub_df.to_csv('submission_golden.csv', index=False)

print("\n☑️ submission_golden.csv successfully created! The dummy penalties are gone! 🏆")
display(golden_sub_df.head(5))

Initiating the GOLDEN RATIO Reranker Pipeline! 🚀


Optimizing Predictions: 100%|██████████| 40/40 [01:36<00:00,  2.42s/it]


☑️ submission_golden.csv successfully created! The dummy penalties are gone! 🏆


,query_id,predicted_citations
0,test_001,4A_575/2018 E. A;4A_27/2012 16.07.2012 E. A;4A...
1,test_002,6B_782/2019 E. 2.2;6B_1209/2015 E. 3.2;6B_1120...
2,test_003,2C_3/2022 E. B;2C_415/2013 E. 5.3;6B_1272/2021...
3,test_004,4F_28/2025 E. 2;4A_365/2024 E. 3;5A_514/2022 E...
4,test_005,2C_24/2018 E. 1.1;2C_662/2013 E. B;6B_838/2018...


## 🔎 Insight 22

1.  **V1 - Only Laws (0.01191):** Our first attempt worked only with the 175k-line laws_df. We didn't touch court decisions at all. Our chance of being "Wrong" was very low (High Precision), but there was a lot we "Missed" (Low Recall). Despite this, the score was at its highest because we weren't making incorrect citations.

2.  **V2 - Big Pool (0.00765):** We dove headfirst into a massive pool of 2.6 Million. We pulled 15 court decisions for every question. Our score plummeted! Why? Because the F1 metric was unforgiving; we brought in plenty of irrelevant court decisions (False Positives), killing our Precision.

3.  **V3 - Federated Search (0.01086):** We searched laws separately and decisions separately, then combined them. We limited decisions to 5. The score recovered!

4.  **V4 - Dummy Citations / Reranker (0.01047):** The reranker was too strict; to avoid leaving gaps, we added irrelevant dummy citations to 11 questions. Our F1 score was penalized for those 11 mistakes.

5.  **V5 - The Golden Ratio (0.01106):** We removed the dummy citations and only took the Top-3 results we were most confident about. The score came very close to the V1 level again, and we had successfully integrated that entire massive pool safely.

# 23. The Final Polish: Adaptive Thresholding 🔧

In [32]:
print("Initiating ADAPTIVE THRESHOLDING Pipeline!🚀")

adaptive_submission_data = []

for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Adaptive Predictions"):
    query_id = row['query_id']
    query_text = row['query']
    
    final_citations = set()
    
    # 1. Exact Match via Regex
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    final_citations.update(exact_matches)
    
    # 2. LLM Query Expansion
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts and translate to German keywords. List abbreviations."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    # 3. Fetch Top 30 Candidates to have a wider pool for Adaptive Reranking
    expanded_query = f"{query_text} {german_keywords}"
    q_emb = embedding_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    top_k_retrieve = 30 
    _, unified_indices = unified_faiss_index.search(q_emb, top_k_retrieve)
    
    candidate_citations = []
    cross_encoder_pairs = []
    
    for idx in unified_indices[0]:
        cit = unified_citations[idx]
        if cit in laws_df['citation'].values:
            text = laws_df[laws_df['citation'] == cit].iloc[0]['text']
        else:
            text = cit
            
        candidate_citations.append(cit)
        cross_encoder_pairs.append([expanded_query, text])
    
    # 4. Adaptive Reranking
    if cross_encoder_pairs:
        rerank_scores = reranker.predict(cross_encoder_pairs)
        scored_candidates = list(zip(candidate_citations, rerank_scores))
        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        
        # Adaptive Logic: Accept anyone within 20% of the top score, AND score must be > 0
        best_score = scored_candidates[0][1]
        
        # We only apply adaptive logic if the best score is actually positive
        if best_score > 0:
            threshold = best_score * 0.80 # 20% drop-off allowed
            
            for cit, score in scored_candidates:
                if score >= threshold:
                    final_citations.add(cit)
                else:
                    break # Since it's sorted, we can stop early 
        else:
            # If the best score is negative, just take the top 1 to avoid Kaggle empty errors
            final_citations.add(scored_candidates[0][0])
                
    predicted_string = ";".join(list(final_citations))
    
    adaptive_submission_data.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

adaptive_sub_df = pd.DataFrame(adaptive_submission_data)
# Safety net
adaptive_sub_df['predicted_citations'] = adaptive_sub_df['predicted_citations'].replace("", "Art. 1 ZGB") 
adaptive_sub_df.to_csv('submission_adaptive.csv', index=False)

print("\n☑️ submission_adaptive.csv successfully created!")
display(adaptive_sub_df.head(5))

Initiating ADAPTIVE THRESHOLDING Pipeline!🚀


Adaptive Predictions: 100%|██████████| 40/40 [01:41<00:00,  2.53s/it]


☑️ submission_adaptive.csv successfully created!


,query_id,predicted_citations
0,test_001,4A_584/2017 09.01.2019 E. A
1,test_002,6B_1209/2015 E. 3.2;8C_400/2016 E. A;6B_1120/2...
2,test_003,H 268/02 21.08.2003 E. A;6B_1272/2021 E. A;4A_...
3,test_004,2C_217/2022 E. A;4F_28/2025 E. 2;2C_54/2025 E....
4,test_005,4A_411/2020 E. A;5C.96/2006 02.08.2006 E. 1;2C...


## 🔎 Insight 23
**Public Score: 0.00936**

We instructed the Reranker model to *"take items up to 20% below the highest score."* However, the output of Cross-Encoder models (like mmarco) is not a "percentage" or "probability" between 0 and 100; these outputs are **Logit** (raw mathematical prediction) values.

**The Mathematical Reality:** In logit space, the difference between 8.0 and 6.4 does **not** represent a 20% drop in probability. When passed through a Sigmoid function, a logit of 8.0 corresponds to a ~99.96% probability, while a logit of 6.4 corresponds to a ~99.8% probability.

**Consequence:** Our model perceived that "20%" threshold so "wide" that it likely gathered 15-20 court decisions per query again, filling the basket with **False Positives** and ultimately ruining our F1 score!

# 24. Hard Negative Mining ⛏️

In [35]:
import random 

print("Initiating HARD NEGATIVE MINING for Fine-Tuning... ⛏️\n")

# 1. Load the Training Dataset
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
print(f"Total Training Queries available: {len(train_df)}")

# Let's extract training triplets for the first 5 queries as a prototype
training_triplets = []

for idx, row in train_df.head(5).iterrows():
    query_text = row['query']
    
    # Handle NaN in gold citations safely
    gold_str = str(row['gold_citations']) if pd.notnull(row['gold_citations']) else ""
    gold_citations = set(gold_str.split(';')) if gold_str else set()
    
    if not gold_citations:
        continue # Skip if no positive examples
        
    # We need the TEXT of the positive citation. Let's pick one random gold citation.
    pos_cit = random.choice(list(gold_citations))
    pos_text = ""
    
    if pos_cit in laws_df['citation'].values:
        pos_text = laws_df[laws_df['citation'] == pos_cit].iloc[0]['text']
    else:
        # If it's a court case, we use the citation itself as context for now
        pos_text = pos_cit 

    # --- MINING THE HARD NEGATIVE ---
    q_emb = embedding_model.encode([query_text])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    # Search the giant 2.6M index
    _, indices = unified_faiss_index.search(q_emb, 15)
    
    neg_cit = None
    neg_text = ""
    
    # Find the highest ranked citation that is NOT in gold_citations
    for i in indices[0]:
        candidate = unified_citations[i]
        if candidate not in gold_citations:
            neg_cit = candidate
            break
            
    if neg_cit:
        if neg_cit in laws_df['citation'].values:
            neg_text = laws_df[laws_df['citation'] == neg_cit].iloc[0]['text']
        else:
            neg_text = neg_cit
            
        training_triplets.append({
            "query": query_text,
            "positive_text": pos_text,
            "negative_text": neg_text
        })

print("\n☑️  Successfully generated Training Triplets!")

# Display the first triplet safely
if training_triplets:
    print("Here is an example of what we will feed to the AI brain:\n")
    example = training_triplets[0]
    print(f"🔸 ANCHOR:\n{example['query'][:200]}...\n")
    print(f"☑️ POSITIVE:\n{example['positive_text'][:200]}...\n")
    print(f"✖️ HARD NEGATIVE:\n{example['negative_text'][:200]}...")
else:
    print("No triplets generated for the first 5 rows (usually due to missing gold citations).")

Initiating HARD NEGATIVE MINING for Fine-Tuning... ⛏️

Total Training Queries available: 1139

☑️  Successfully generated Training Triplets!
Here is an example of what we will feed to the AI brain:

🔸 ANCHOR:
Die A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) ein Recyclingunternehmen. Das Unternehmen verarbeitet vorwiegend Metallschrott. Dieser wird im Aussenbereich d...

☑️ POSITIVE:
Art. 10a Abs. 1 UVG...

✖️ HARD NEGATIVE:
1C_397/2013 E. A...


## 🔎 Insight 24

- **Anchor (Query):** A German text about a recycling factory (A AG) processing metal scrap.

- **Hard Negative (Wrong Decision):** Our system previously retrieved court decision number 1C_397/2013. Why? Because that court decision likely also contained words like "environmental law, waste facility, zoning plan." Our vector model was misled by this word similarity (semantics) into thinking it was the "best answer."

- **Positive (Correct Law):** The actual answer was the Swiss Accident Insurance Act (Art. 10a Para. 1 UVG)! Apparently, a very specific rule related to an occupational accident or disease at the factory was needed.

When we feed this triplet to the model, a mathematical process called **Contrastive Learning** runs in the background. In the **Vector Space**, the model will magnetically pull the "Anchor" and "Positive" closer together while forcefully pushing that confusing "Hard Negative" far away.

# 25. The AI Gym: Fine-Tuning MiniLM 🏋️

In [36]:
from sentence_transformers import InputExample, losses, SentenceTransformer
from torch.utils.data import DataLoader

print("--- PHASE 1: Mining Hard Negatives for FULL Training Set --- ⛏️")
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
training_examples = []

# Loop over the entire training set
for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Mining Triplets"):
    query_text = row['query']
    
    gold_str = str(row['gold_citations']) if pd.notnull(row['gold_citations']) else ""
    gold_citations = set(gold_str.split(';')) if gold_str else set()
    
    if not gold_citations:
        continue 
        
    pos_cit = random.choice(list(gold_citations))
    pos_text = pos_cit
    if pos_cit in laws_df['citation'].values:
        pos_text = laws_df[laws_df['citation'] == pos_cit].iloc[0]['text']

    # Find Hard Negative (Zorlu Negatifi Bul)
    q_emb = embedding_model.encode([query_text], show_progress_bar=False)
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    _, indices = unified_faiss_index.search(q_emb, 15)
    
    neg_cit = None
    neg_text = ""
    
    for i in indices[0]:
        candidate = unified_citations[i]
        if candidate not in gold_citations:
            neg_cit = candidate
            break
            
    if neg_cit:
        if neg_cit in laws_df['citation'].values:
            neg_text = laws_df[laws_df['citation'] == neg_cit].iloc[0]['text']
        else:
            neg_text = neg_cit
            
        # Format required by sentence-transformers
        # (sentence-transformers tarafından istenen format)
        training_examples.append(InputExample(texts=[query_text, pos_text, neg_text]))

print(f"\n☑️  Successfully mined {len(training_examples)} training triplets!")

print("\n--- PHASE 2: Fine-Tuning the Embedding Model --- 🏋️‍♂️")
# 1. Create a DataLoader
# Batch size 16 is good for P100 GPU without running out of memory
train_dataloader = DataLoader(training_examples, shuffle=True, batch_size=16)

# 2. Define the Loss Function
# MultipleNegativesRankingLoss is perfect for Triplet data (Anchor, Positive, Negative)
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# 3. Train the Model!
print("Starting training... This will take a few minutes.")
start_time = time.time()

embedding_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1, # 1 epoch is usually enough to inject domain knowledge without overfitting
    warmup_steps=int(len(train_dataloader) * 0.1), # 10% warmup
    show_progress_bar=True
)

print(f"\n☑️  Fine-Tuning Completed in {(time.time() - start_time) / 60:.2f} minutes!")

# 4. Save our newly minted Swiss Legal Model!
tuned_model_path = "swiss-legal-minilm-v1"
embedding_model.save(tuned_model_path)
print(f"Model saved locally to: {tuned_model_path}")

--- PHASE 1: Mining Hard Negatives for FULL Training Set --- ⛏️


Mining Triplets: 100%|██████████| 1139/1139 [07:00<00:00,  2.71it/s]



☑️  Successfully mined 1139 training triplets!

--- PHASE 2: Fine-Tuning the Embedding Model --- 🏋️‍♂️
Starting training... This will take a few minutes.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss



☑️  Fine-Tuning Completed in 0.29 minutes!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved locally to: swiss-legal-minilm-v1


## 🔎 Insight 25
We now have a brand new brain (Compass) called `swiss-legal-minilm-v1`, entirely yours, which has learned the nuances of Swiss law (the Anchor-Positive-Negative relationship).

However, a critical data science rule comes into play here:

- **Stale Index:** The FAISS indexes (Maps) we created in Steps 14 and 17 were built using the "old and untrained" version of our model.

- If we use our new, smart brain to search on that old map of vectors, we will get mathematically completely misaligned results. Because the model's weights have changed, the numerical values it assigns to words have also changed entirely!

To test the power of this new brain, we must redraw our map from scratch using this new brain.

# 26. The Ultimate A/B Test 🧪

In [37]:
from sentence_transformers import SentenceTransformer

print("Loading our Custom Fine-Tuned Model! 🧠")
# Load the model we just trained!
tuned_model = SentenceTransformer('swiss-legal-minilm-v1', device='cuda')

print("\nRe-encoding the Laws Corpus (175k rows)...")
start_time = time.time()

# Encode laws with the NEW model
laws_texts = laws_df['text'].fillna("").tolist()
tuned_embeddings = tuned_model.encode(laws_texts, show_progress_bar=True)
tuned_embeddings = np.array(tuned_embeddings).astype('float32')
faiss.normalize_L2(tuned_embeddings)

# Build a NEW FAISS index specifically for the tuned embeddings
embedding_dim = tuned_embeddings.shape[1]
faiss_index_tuned = faiss.IndexFlatIP(embedding_dim)
faiss_index_tuned.add(tuned_embeddings)

print(f"FAISS Tuned Index ready! Time: {time.time() - start_time:.2f}s.")

# --- Run the Baseline Submission with the Tuned Model ---
print("\nInitiating Submission Pipeline with Tuned Model... 🚀")
tuned_submission_data = []

for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Tuned Predictions"):
    query_id = row['query_id']
    query_text = row['query']
    
    final_citations = set()
    
    # 1. Regex Exact Match
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    final_citations.update(exact_matches)
    
    # 2. LLM Query Expansion
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts and translate to German keywords. List abbreviations."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    # 3. Dense Search with TUNED MODEL
    expanded_query = f"{query_text} {german_keywords}"
    
    # Encode the query with the TUNED model!
    q_emb = tuned_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    # Let's get top 5 laws
    _, indices = faiss_index_tuned.search(q_emb, 5)
    
    for idx in indices[0]:
        final_citations.add(laws_df.iloc[idx]['citation'])
        
    # Format for Kaggle
    predicted_string = ";".join(list(final_citations))
    
    tuned_submission_data.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

# Save and explicitly fill NaNs just in case
tuned_sub_df = pd.DataFrame(tuned_submission_data)
tuned_sub_df['predicted_citations'] = tuned_sub_df['predicted_citations'].replace("", "Art. 1 ZGB") 
tuned_sub_df.to_csv('submission_finetuned_laws.csv', index=False)

print("\n☑️ 'submission_finetuned_laws.csv' successfully created!")
display(tuned_sub_df.head(5))

Loading our Custom Fine-Tuned Model! 🧠


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Re-encoding the Laws Corpus (175k rows)...


Batches:   0%|          | 0/5498 [00:00<?, ?it/s]

FAISS Tuned Index ready! Time: 133.28s.

Initiating Submission Pipeline with Tuned Model... 🚀


Tuned Predictions: 100%|██████████| 40/40 [01:20<00:00,  2.02s/it]


☑️ 'submission_finetuned_laws.csv' successfully created!


,query_id,predicted_citations
0,test_001,Art. 11 Abs. 1 VEleS;Art. 10 Abs. 3 VTE;Art. 5...
1,test_002,Art. 94 Abs. 4 SVG;Art. 63 Abs. 3 VRV;Art. 70 ...
2,test_003,Art. 8a STUG;Art. 3 Abs. 1 STUG;Art. 11 Abs. 1...
3,test_004,Art. 67a Abs. 2 KVV;Art. 95 Abs. 1 PGesV;Art. ...
4,test_005,Art. 28 Abs. 2 AVV;Art. 49 Abs. 4 URG;Art. 36 ...


## 🔎 Insight 26 
**Public Score:** 0.01454

- **Untrained Model (Previous):** The model matched the word "Tutukluluk (Pre-trial detention)" in the question to an irrelevant "Tutukluluk" article in the Swiss Civil Code. It only understood the general meaning of the words.

- **Trained Model (Current - swiss-legal-minilm):** Using **Hard Negative** mining, we taught the model: "Yes, both contain 'tutukluluk,' but this is a question about Criminal Procedure (StPO), not the Civil Code (ZGB)!" The model memorized which legal articles correspond to the German translations of English legal terms. It effectively repositioned the word clusters in the **Vector Space**.

# 27. The Ultimate Index: Re-encoding the Giant 🌋

In [38]:
print("Awakening the Giant with our NEW Brain! 🧠🚀")

# 1. Load our fine-tuned champion model 
tuned_model = SentenceTransformer('swiss-legal-minilm-v1', device='cuda')

# 2. Create the ULTIMATE FAISS Index 
embedding_dim = 384
ultimate_faiss_index = faiss.IndexFlatIP(embedding_dim)
ultimate_citations = [] # Master list for mapping indices to citations

# --- A. Encode Laws ---
print("\n[1/2] Encoding Laws (175k rows)...")
start_time = time.time()
laws_texts = laws_df['text'].fillna("").tolist()
laws_embeddings = tuned_model.encode(laws_texts, show_progress_bar=True)
laws_embeddings = np.array(laws_embeddings).astype('float32')
faiss.normalize_L2(laws_embeddings)

ultimate_faiss_index.add(laws_embeddings)
ultimate_citations.extend(laws_df['citation'].tolist())
print(f"Laws added! Time: {time.time() - start_time:.2f}s. Total vectors: {ultimate_faiss_index.ntotal}")

# --- B. Encode Court Considerations ---
print("\n[2/2] Encoding Court Cases (2.4M rows)... This will take ~55 minutes. Grab a coffee! ☕")

chunk_size = 50000
court_file_path = f'{DATA_DIR}/court_considerations.csv'
chunk_count = 0
total_court_time = 0

# Open the giant CSV in chunks to save RAM
for chunk in pd.read_csv(court_file_path, chunksize=chunk_size):
    chunk_start = time.time()
    chunk_count += 1
    
    texts = chunk['text'].fillna("").tolist()
    citations = chunk['citation'].tolist()
    
    # Encode with the TUNED model!
    embeddings = tuned_model.encode(texts, show_progress_bar=False) 
    embeddings = np.array(embeddings).astype('float32')
    faiss.normalize_L2(embeddings)
    
    ultimate_faiss_index.add(embeddings)
    ultimate_citations.extend(citations)
    
    chunk_time = time.time() - chunk_start
    total_court_time += chunk_time
    
    if chunk_count % 5 == 0:
        print(f"Processed chunk {chunk_count} ({chunk_count * chunk_size} rows)... Total vectors: {ultimate_faiss_index.ntotal}")

print(f"\n☑️ THE ULTIMATE INDEX IS READY!")
print(f"Total time for Court Cases: {total_court_time / 60:.2f} minutes.")
print(f"Final FAISS Index Size: {ultimate_faiss_index.ntotal} vectors.")

Awakening the Giant with our NEW Brain! 🧠🚀


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[1/2] Encoding Laws (175k rows)...


Batches:   0%|          | 0/5498 [00:00<?, ?it/s]

Laws added! Time: 134.94s. Total vectors: 175933

[2/2] Encoding Court Cases (2.4M rows)... This will take ~55 minutes. Grab a coffee! ☕
Processed chunk 5 (250000 rows)... Total vectors: 425933
Processed chunk 10 (500000 rows)... Total vectors: 675933
Processed chunk 15 (750000 rows)... Total vectors: 925933
Processed chunk 20 (1000000 rows)... Total vectors: 1175933
Processed chunk 25 (1250000 rows)... Total vectors: 1425933
Processed chunk 30 (1500000 rows)... Total vectors: 1675933
Processed chunk 35 (1750000 rows)... Total vectors: 1925933
Processed chunk 40 (2000000 rows)... Total vectors: 2175933
Processed chunk 45 (2250000 rows)... Total vectors: 2425933
Processed chunk 50 (2500000 rows)... Total vectors: 2652248

☑️ THE ULTIMATE INDEX IS READY!
Total time for Court Cases: 56.95 minutes.
Final FAISS Index Size: 2652248 vectors.


# 28. The Grand Finale: Ultimate Submission 🏆

In [39]:
print("Initiating the ULTIMATE GRAND FINALE Submission! 🏆🚀")

grand_finale_submission = []

for row_idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Forging the Champion File"):
    query_id = row['query_id']
    query_text = row['query']
    
    final_citations = set()
    
    # --- 1. Rule-Based Agent (Regex) ---
    extracted_candidates = extract_and_clean_citations(query_text)
    exact_matches = laws_df[laws_df['citation'].isin(extracted_candidates)]['citation'].tolist()
    final_citations.update(exact_matches)
    
    # --- 2. Translation Agent (LLM Query Expansion) ---
    prompt_messages = [
        {"role": "system", "content": "You are a Swiss lawyer. Extract core legal concepts and translate to German keywords. List abbreviations."},
        {"role": "user", "content": f"Query: {query_text}\n\nKeywords:"}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    outputs = agent_pipe(prompt, do_sample=False, max_new_tokens=50)
    german_keywords = outputs[0]["generated_text"][len(prompt):].strip()
    
    # --- 3. Retrieval Agent (Fine-Tuned FAISS Search) ---
    expanded_query = f"{query_text} {german_keywords}"
    
    # IMPORTANT: We encode with the TUNED model!
    q_emb = tuned_model.encode([expanded_query])
    q_emb = np.array(q_emb).astype('float32')
    faiss.normalize_L2(q_emb)
    
    # Fetch top 15 from the 2.6M ULTIMATE INDEX
    top_k_retrieve = 15
    _, unified_indices = ultimate_faiss_index.search(q_emb, top_k_retrieve)
    
    candidate_citations = []
    cross_encoder_pairs = []
    
    for idx in unified_indices[0]:
        cit = ultimate_citations[idx]
        if cit in laws_df['citation'].values:
            text = laws_df[laws_df['citation'] == cit].iloc[0]['text']
        else:
            text = cit # Use citation ID as context for court cases to save RAM
            
        candidate_citations.append(cit)
        cross_encoder_pairs.append([expanded_query, text])
    
    # --- 4. Reranking Agent (Cross-Encoder Top-3 Logic) ---
    if cross_encoder_pairs:
        rerank_scores = reranker.predict(cross_encoder_pairs)
        
        scored_candidates = list(zip(candidate_citations, rerank_scores))
        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        
        # Take the top 3 to keep Precision high while exploring Court Cases!
        for cit, score in scored_candidates[:3]:
            final_citations.add(cit)
            
    # --- Format for Kaggle ---
    predicted_string = ";".join(list(final_citations))
    
    grand_finale_submission.append({
        'query_id': query_id,
        'predicted_citations': predicted_string
    })

# Save and apply the safety net for the Kaggle NaN bug
finale_df = pd.DataFrame(grand_finale_submission)
finale_df['predicted_citations'] = finale_df['predicted_citations'].replace("", "Art. 1 ZGB") 
finale_df.to_csv('submission_grand_finale.csv', index=False)

print("\n☑️ submission_grand_finale.csv successfully created! The masterpiece is done! 🏆")
display(finale_df.head(5))

Initiating the ULTIMATE GRAND FINALE Submission! 🏆🚀


Forging the Champion File: 100%|██████████| 40/40 [01:40<00:00,  2.52s/it]


☑️ submission_grand_finale.csv successfully created! The masterpiece is done! 🏆


,query_id,predicted_citations
0,test_001,4A_149/2013 E. 5.4.1;2C_522/2019 E. 3.3.1;4A_4...
1,test_002,6B_651/2012 E. 3.2;6B_782/2019 E. 2.2;Art. 59 ...
2,test_003,2C_415/2013 E. 5.3;4A_609/2018 E. 4;2C_228/201...
3,test_004,4A_365/2024 E. 3;2C_848/2022 E. B;5A_514/2022 ...
4,test_005,4A_88/2018 E. 2;6B_838/2018 E. 5.2.1;4A_25/202...


## 🔎 Insight 28
**Public Score: 0.01384**

This long and arduous journey is actually perfect proof of the most fundamental philosophy of machine learning, the principle of Occam's Razor: *"If two different models yield similar results, prefer the simpler one."*

So, why couldn't the 2.6 million dataset beat the mere 175 thousand-row law dataset?

- **Noise Injection:** Court decisions are so long and specific that the model takes a single word from the question and gets stuck on a massive court ruling where the same word appears, but the context is completely different.

- **Top-3 Filter Missed Opportunities:** To keep the score from dropping, we retrieved a maximum of 3 results (Top-3) per question. If the model spends two of these three rights on incorrect court rulings, it cannot include the short, golden laws it actually needs to find in the list. When we searched only for laws (0.01454), the majority of the 5 laws we retrieved were correct.

- **The Cross-Encoder Pitfall:** The `mmarco` reranker model we used was general-purpose. We never trained it on Swiss law (we only trained MiniLM). Consequently, the judge (Reranker) making the final decision was actually ignorant of Swiss law!

# 29. 🏆 Conclusion

Building an end-to-end legal Retrieval-Augmented Generation (RAG) system is rarely a straight line. Throughout this notebook, we evolved our pipeline from a simple lexical search into a highly advanced, AI-driven architectural marvel. 

We processed over **2.6 million documents** covering the entirety of Swiss Law and Supreme Court decisions, navigating severe memory constraints, Kaggle environment bugs, and the complexities of multilingual legal text.

### 🌟 Key Architectural Milestones Achieved:
1. **Agentic Query Expansion (HyDE):** We deployed a 1.5B parameter LLM to dynamically translate complex English legal scenarios into precise German keywords and abbreviations.
2. **Hybrid & Federated Search:** We separated our retrieval engines, recognizing early on that long-winded court decisions easily drown out short, concise statutory laws in a shared vector space.
3. **Two-Stage Retrieval:** We implemented a rapid **Bi-Encoder (MiniLM)** with FAISS for semantic recall, followed by a ruthless **Cross-Encoder (mmarco)** to rerank candidates and filter out false positives.
4. **Domain-Specific Fine-Tuning:** The ultimate breakthrough. By mining **Hard Negatives** from our own mistakes, we fine-tuned our embedding model using *Multiple Negatives Ranking Loss (MNRL)*. This 15-second training process boosted our baseline score by over **22%**!



### 🧠 The Ultimate Insight: Occam's Razor in Machine Learning
Our highest Public Score (`0.01454`) was achieved not by throwing the entire 2.6M document corpus at the model, but by using our newly **fine-tuned model strictly on the 175k statutory laws**. 

Why? Because the 2.4 million court cases introduced immense **Semantic Noise**. Without meticulously chunking those lengthy court decisions and explicitly fine-tuning the *Cross-Encoder* on Swiss legal nuances, the system struggled to distinguish between a case *mentioning* a law and a case that acts as the *ground truth precedent* for the query. **Bigger data isn't always better; cleaner, domain-aligned data is.**

### 🚀 Future Work & Next Steps
If we had unconstrained compute and time, the next logical steps to dominate the leaderboard would be:
* **Chunking Strategy:** Break the massive 2.5GB court decisions into smaller, overlapping 256-token chunks to prevent the "Length Bias" in dense retrieval.
* **Fine-Tuning the Cross-Encoder:** We fine-tuned the Bi-Encoder, but the final judge (the Cross-Encoder) was still a zero-shot model. Training it on Swiss Law would perfectly align the reranking phase.
* **BM25 Integration:** Utilizing a sparse retrieval algorithm (like Elasticsearch or a disk-based BM25 implementation) alongside FAISS to catch exact keyword matches that vectors sometimes miss.

Thank you for following along on this incredible Data Science and AI Engineering marathon. We didn't just write code; we built a brain. 🇨🇭